In [ ]:
# =============================================================================
# Evaluate fix rates and resample WTD data from approximately 3-hour fixes
# to approximately 6-hour fixes for comparison with the mule deer study
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 1. File paths
# =============================================================================

BASE_PATH = Path("<LOCAL_PATH>")

DATASETS = {
    "Train": "CWD_negative_training_with_seasons2.csv",
    "Test": "CWD_positive_testing_with_seasons2.csv",
}

OUTPUT_DIR = BASE_PATH / "six_hour_resampled"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 2. User settings
# =============================================================================

ID_COL = "id"
DATE_COL = "date"
STATUS_COL = "cwd_status"

# Target sampling interval used in the mule deer study
TARGET_INTERVAL_HOURS = 6

# A target time will only be matched to an observed fix when it falls within
# this tolerance. For nominal 3-hour data, 1.5 hours is half the original
# sampling interval.
MATCH_TOLERANCE_HOURS = 1.5

# Require at least this many fixes on a calendar day when describing
# approximately complete 6-hour days.
EXPECTED_FIXES_PER_DAY = 4


# =============================================================================
# 3. Basic cleaning
# =============================================================================

def clean_datetime_data(df, label):
    """
    Standardize ID and datetime columns and remove unusable records.
    """

    df = df.copy()

    required_columns = [ID_COL, DATE_COL]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{label} is missing required columns: {missing_columns}"
        )

    original_n = len(df)

    # Preserve IDs as strings
    df[ID_COL] = df[ID_COL].astype(str).str.strip()

    # Convert timestamps
    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="coerce"
    )

    invalid_dates = df[DATE_COL].isna().sum()
    missing_ids = (
        df[ID_COL].isna() |
        df[ID_COL].eq("") |
        df[ID_COL].eq("nan")
    ).sum()

    # Remove unusable rows
    df = df.dropna(subset=[DATE_COL])
    df = df[
        df[ID_COL].notna() &
        df[ID_COL].ne("") &
        df[ID_COL].ne("nan")
    ].copy()

    # Remove exact duplicate ID/timestamp combinations
    duplicate_count = df.duplicated(
        subset=[ID_COL, DATE_COL],
        keep="first"
    ).sum()

    df = df.drop_duplicates(
        subset=[ID_COL, DATE_COL],
        keep="first"
    )

    df = df.sort_values(
        [ID_COL, DATE_COL]
    ).reset_index(drop=True)

    print(f"\n{'=' * 80}")
    print(f"{label}: DATETIME CLEANING")
    print(f"{'=' * 80}")
    print(f"Original rows:               {original_n:,}")
    print(f"Invalid timestamps:          {invalid_dates:,}")
    print(f"Missing/invalid IDs:         {missing_ids:,}")
    print(f"Duplicate ID/timestamps:     {duplicate_count:,}")
    print(f"Rows retained after cleaning:{len(df):,}")

    return df


# =============================================================================
# 4. ID, status, and fix-count summary
# =============================================================================

def print_unique_ids_and_status(df, label):
    """
    Print unique IDs, CWD statuses, and total fix counts.
    """

    ids = sorted(df[ID_COL].dropna().unique())

    print(f"\n{'=' * 80}")
    print(f"{label}: ID AND STATUS SUMMARY")
    print(f"{'=' * 80}")

    print(f"\n{len(ids)} unique IDs:")
    print(ids)

    if STATUS_COL in df.columns:
        status_values = sorted(
            df[STATUS_COL].dropna().astype(str).unique()
        )

        print(f"\nObserved status values: {status_values}")

        positive_ids = sorted(
            df.loc[
                df[STATUS_COL].astype(str).str.lower().eq("positive"),
                ID_COL
            ].unique()
        )

        negative_ids = sorted(
            df.loc[
                df[STATUS_COL].astype(str).str.lower().eq("negative"),
                ID_COL
            ].unique()
        )

        print(
            f"CWD-positive IDs ({len(positive_ids)}): "
            f"{positive_ids}"
        )

        print(
            f"CWD-negative IDs ({len(negative_ids)}): "
            f"{negative_ids}"
        )

    fix_counts = df.groupby(ID_COL).size().sort_index()

    print("\nNumber of GPS fixes per ID:")
    print(fix_counts.to_string())

    print("\nFix-count summary:")
    print(f"  Minimum: {fix_counts.min():,.0f}")
    print(f"  Maximum: {fix_counts.max():,.0f}")
    print(f"  Median:  {fix_counts.median():,.1f}")
    print(f"  Mean:    {fix_counts.mean():,.1f}")


# =============================================================================
# 5. Detailed fix-rate EDA
# =============================================================================

def summarize_fix_rate(df, label):
    """
    Summarize tracking duration, intervals, and daily fix counts.
    """

    working = df.sort_values(
        [ID_COL, DATE_COL]
    ).copy()

    # -------------------------------------------------------------------------
    # Tracking duration
    # -------------------------------------------------------------------------

    tracking_summary = (
        working.groupby(ID_COL)[DATE_COL]
        .agg(
            first_fix="min",
            last_fix="max",
            total_fixes="size"
        )
        .reset_index()
    )

    tracking_summary["duration_days"] = (
        tracking_summary["last_fix"] -
        tracking_summary["first_fix"]
    ).dt.total_seconds() / 86_400

    # Inclusive calendar coverage
    tracking_summary["inclusive_days"] = (
        tracking_summary["duration_days"] + 1
    )

    tracking_summary["fixes_per_tracking_day"] = (
        tracking_summary["total_fixes"] /
        tracking_summary["inclusive_days"]
    )

    # -------------------------------------------------------------------------
    # Time intervals between consecutive fixes
    # -------------------------------------------------------------------------

    working["interval_hours"] = (
        working.groupby(ID_COL)[DATE_COL]
        .diff()
        .dt.total_seconds()
        .div(3600)
    )

    positive_intervals = working.loc[
        working["interval_hours"] > 0,
        "interval_hours"
    ]

    # -------------------------------------------------------------------------
    # Fixes per calendar day
    # -------------------------------------------------------------------------

    working["calendar_date"] = working[DATE_COL].dt.date

    daily_counts = (
        working.groupby([ID_COL, "calendar_date"])
        .size()
        .rename("fixes_per_day")
        .reset_index()
    )

    per_id_daily_summary = (
        daily_counts.groupby(ID_COL)["fixes_per_day"]
        .agg(
            number_of_observed_days="size",
            mean_fixes_per_day="mean",
            median_fixes_per_day="median",
            minimum_fixes_per_day="min",
            maximum_fixes_per_day="max"
        )
        .reset_index()
    )

    # Merge major summaries
    id_summary = tracking_summary.merge(
        per_id_daily_summary,
        on=ID_COL,
        how="left"
    )

    print(f"\n{'=' * 80}")
    print(f"{label}: FIX-RATE EDA")
    print(f"{'=' * 80}")

    print(
        f"Overall date range: "
        f"{working[DATE_COL].min()} to {working[DATE_COL].max()}"
    )

    print("\nTracking duration across IDs:")
    print(
        f"  Minimum: {tracking_summary['duration_days'].min():.2f} days"
    )
    print(
        f"  Maximum: {tracking_summary['duration_days'].max():.2f} days"
    )
    print(
        f"  Median:  {tracking_summary['duration_days'].median():.2f} days"
    )
    print(
        f"  Mean:    {tracking_summary['duration_days'].mean():.2f} days"
    )

    if not positive_intervals.empty:
        print("\nIntervals between consecutive fixes:")
        print(
            f"  Minimum: {positive_intervals.min():.2f} hours"
        )
        print(
            f"  25th percentile: "
            f"{positive_intervals.quantile(0.25):.2f} hours"
        )
        print(
            f"  Median:  {positive_intervals.median():.2f} hours"
        )
        print(
            f"  Mean:    {positive_intervals.mean():.2f} hours"
        )
        print(
            f"  75th percentile: "
            f"{positive_intervals.quantile(0.75):.2f} hours"
        )
        print(
            f"  Maximum: {positive_intervals.max():.2f} hours"
        )

        approximately_3_hour = positive_intervals.between(
            2.5, 3.5
        ).mean() * 100

        approximately_6_hour = positive_intervals.between(
            5.5, 6.5
        ).mean() * 100

        print(
            f"  Intervals between 2.5 and 3.5 hours: "
            f"{approximately_3_hour:.1f}%"
        )
        print(
            f"  Intervals between 5.5 and 6.5 hours: "
            f"{approximately_6_hour:.1f}%"
        )

    print("\nDaily fix-count distribution across all ID-days:")
    print(daily_counts["fixes_per_day"].describe().to_string())

    print("\nMean daily fix rate across IDs:")
    print(
        f"  Minimum: "
        f"{per_id_daily_summary['mean_fixes_per_day'].min():.2f}"
    )
    print(
        f"  Maximum: "
        f"{per_id_daily_summary['mean_fixes_per_day'].max():.2f}"
    )
    print(
        f"  Median:  "
        f"{per_id_daily_summary['mean_fixes_per_day'].median():.2f}"
    )
    print(
        f"  Mean:    "
        f"{per_id_daily_summary['mean_fixes_per_day'].mean():.2f}"
    )

    print("\nPer-ID sampling summary:")
    columns_to_print = [
        ID_COL,
        "total_fixes",
        "duration_days",
        "fixes_per_tracking_day",
        "mean_fixes_per_day",
        "median_fixes_per_day",
        "minimum_fixes_per_day",
        "maximum_fixes_per_day"
    ]

    print(
        id_summary[columns_to_print]
        .round(2)
        .to_string(index=False)
    )

    return {
        "tracking_summary": tracking_summary,
        "daily_counts": daily_counts,
        "id_summary": id_summary,
        "interval_data": working[
            [ID_COL, DATE_COL, "interval_hours"]
        ].copy()
    }


# =============================================================================
# 6. Resample one deer to approximately six-hour intervals
#    while guaranteeing that every ID is retained
# =============================================================================

def resample_single_id_to_six_hours(
    group,
    interval_hours=6,
    tolerance_hours=2.0
):
    """
    Resample one animal to an approximately regular six-hour schedule.

    Every ID is guaranteed to retain at least one observation.

    Primary method:
        Select the closest original fix to each six-hour target time.

    Fallback method:
        Retain every second observation, which converts nominal three-hour
        sampling to approximately six-hour sampling.

    Notes
    -----
    - The original observation time is retained in `original_fix_time`.
    - The analytical `date` remains the actual observed timestamp.
    - `target_six_hour_time` records the intended six-hour grid time.
    """

    group = (
        group.sort_values(DATE_COL)
        .drop_duplicates(subset=[DATE_COL], keep="first")
        .copy()
    )

    if group.empty:
        return group

    animal_id = group[ID_COL].iloc[0]

    # -------------------------------------------------------------------------
    # Guarantee retention for IDs with only one observation
    # -------------------------------------------------------------------------

    if len(group) == 1:
        result = group.copy()

        result["original_fix_time"] = result[DATE_COL]
        result["target_six_hour_time"] = result[DATE_COL]
        result["time_offset_minutes"] = 0.0
        result["resampling_interval_hours"] = interval_hours
        result["resampling_tolerance_hours"] = tolerance_hours
        result["resampling_method"] = "single_fix_retained"

        return result.reset_index(drop=True)

    # -------------------------------------------------------------------------
    # Create a six-hour target grid
    # -------------------------------------------------------------------------

    first_time = group[DATE_COL].min()
    last_time = group[DATE_COL].max()

    # Anchor the grid to the first actual fix.
    # This avoids forcing animals onto midnight-based clock times.
    target_times = pd.date_range(
        start=first_time,
        end=last_time,
        freq=f"{interval_hours}h"
    )

    target_grid = pd.DataFrame({
        "target_six_hour_time": target_times
    })

    source = group.copy()
    source["original_fix_time"] = source[DATE_COL]
    source["_source_row_id"] = np.arange(len(source))

    source = source.sort_values("original_fix_time")
    target_grid = target_grid.sort_values("target_six_hour_time")

    # -------------------------------------------------------------------------
    # First attempt: nearest observation within the preferred tolerance
    # -------------------------------------------------------------------------

    matched = pd.merge_asof(
        target_grid,
        source,
        left_on="target_six_hour_time",
        right_on="original_fix_time",
        direction="nearest",
        tolerance=pd.Timedelta(hours=tolerance_hours)
    )

    matched = matched.dropna(
        subset=["_source_row_id"]
    ).copy()

    # -------------------------------------------------------------------------
    # If tolerance matching found nothing, match without a tolerance.
    # This guarantees that irregularly sampled IDs are not discarded.
    # -------------------------------------------------------------------------

    if matched.empty:

        matched = pd.merge_asof(
            target_grid,
            source,
            left_on="target_six_hour_time",
            right_on="original_fix_time",
            direction="nearest"
        )

        matched = matched.dropna(
            subset=["_source_row_id"]
        ).copy()

        resampling_method = "nearest_without_tolerance"

    else:
        resampling_method = "nearest_with_tolerance"

    # -------------------------------------------------------------------------
    # Calculate distance between actual and target timestamps
    # -------------------------------------------------------------------------

    if not matched.empty:

        matched["time_offset_minutes"] = (
            matched["original_fix_time"] -
            matched["target_six_hour_time"]
        ).dt.total_seconds().abs() / 60

        # A single observed fix should not be assigned to multiple target times.
        # Retain the target time closest to that observation.
        matched = (
            matched.sort_values(
                ["_source_row_id", "time_offset_minutes"]
            )
            .drop_duplicates(
                subset=["_source_row_id"],
                keep="first"
            )
            .sort_values("original_fix_time")
            .copy()
        )

    # -------------------------------------------------------------------------
    # Final fallback: retain every second observation
    # -------------------------------------------------------------------------

    if matched.empty:

        matched = group.iloc[::2].copy()

        if matched.empty:
            matched = group.iloc[[0]].copy()

        matched["original_fix_time"] = matched[DATE_COL]
        matched["target_six_hour_time"] = matched[DATE_COL]
        matched["time_offset_minutes"] = 0.0

        resampling_method = "every_second_fix_fallback"

    # -------------------------------------------------------------------------
    # Preserve actual timestamps for movement calculations
    # -------------------------------------------------------------------------

    matched[ID_COL] = animal_id

    # Use actual observation time, not the artificial grid time.
    matched[DATE_COL] = matched["original_fix_time"]

    matched["resampling_interval_hours"] = interval_hours
    matched["resampling_tolerance_hours"] = tolerance_hours
    matched["resampling_method"] = resampling_method

    matched = matched.drop(
        columns=["_source_row_id"],
        errors="ignore"
    )

    return matched.reset_index(drop=True)


# =============================================================================
# 7. Resample the complete dataset without dropping IDs
# =============================================================================

def resample_dataset_to_six_hours(
    df,
    label,
    interval_hours=6,
    tolerance_hours=2.0
):
    """
    Apply approximately six-hour resampling separately to each deer.

    This function verifies that all original IDs remain in the output.
    """

    original_ids = set(
        df[ID_COL].dropna().astype(str).unique()
    )

    resampled_groups = []

    for animal_id, group in df.groupby(ID_COL, sort=True):

        resampled_group = resample_single_id_to_six_hours(
            group=group,
            interval_hours=interval_hours,
            tolerance_hours=tolerance_hours
        )

        # Absolute final safeguard
        if resampled_group.empty:

            print(
                f"Warning: fallback required for ID {animal_id}. "
                f"Retaining its first observation."
            )

            resampled_group = group.sort_values(DATE_COL).iloc[[0]].copy()

            resampled_group["original_fix_time"] = (
                resampled_group[DATE_COL]
            )

            resampled_group["target_six_hour_time"] = (
                resampled_group[DATE_COL]
            )

            resampled_group["time_offset_minutes"] = 0.0
            resampled_group["resampling_interval_hours"] = interval_hours
            resampled_group["resampling_tolerance_hours"] = tolerance_hours
            resampled_group["resampling_method"] = "forced_first_fix"

        resampled_groups.append(resampled_group)

    if not resampled_groups:
        raise ValueError(
            f"No observations were retained for {label}."
        )

    resampled = pd.concat(
        resampled_groups,
        ignore_index=True
    )

    resampled[ID_COL] = (
        resampled[ID_COL]
        .astype(str)
        .str.strip()
    )

    resampled = (
        resampled.sort_values([ID_COL, DATE_COL])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Verify ID retention
    # -------------------------------------------------------------------------

    resampled_ids = set(
        resampled[ID_COL].dropna().astype(str).unique()
    )

    missing_ids = sorted(original_ids - resampled_ids)
    added_ids = sorted(resampled_ids - original_ids)

    print(f"\n{'=' * 80}")
    print(f"{label}: ID RETENTION CHECK")
    print(f"{'=' * 80}")

    print(f"Original unique IDs:  {len(original_ids)}")
    print(f"Resampled unique IDs: {len(resampled_ids)}")

    if missing_ids:
        raise RuntimeError(
            f"The following IDs were unexpectedly lost: {missing_ids}"
        )
    else:
        print("All original IDs were retained.")

    if added_ids:
        raise RuntimeError(
            f"Unexpected IDs appeared in the output: {added_ids}"
        )

    # -------------------------------------------------------------------------
    # Fix-count comparison
    # -------------------------------------------------------------------------

    original_counts = (
        df.groupby(ID_COL)
        .size()
        .rename("original_fixes")
    )

    resampled_counts = (
        resampled.groupby(ID_COL)
        .size()
        .rename("six_hour_fixes")
    )

    comparison = pd.concat(
        [original_counts, resampled_counts],
        axis=1
    ).fillna(0)

    comparison["original_fixes"] = (
        comparison["original_fixes"].astype(int)
    )

    comparison["six_hour_fixes"] = (
        comparison["six_hour_fixes"].astype(int)
    )

    comparison["retained_percent"] = (
        comparison["six_hour_fixes"] /
        comparison["original_fixes"] *
        100
    )

    print(f"\n{'=' * 80}")
    print(f"{label}: SIX-HOUR RESAMPLING RESULTS")
    print(f"{'=' * 80}")

    print(f"Original rows:     {len(df):,}")
    print(f"Six-hour rows:     {len(resampled):,}")
    print(
        f"Overall retention: "
        f"{len(resampled) / len(df) * 100:.1f}%"
    )

    print("\nRetention by ID:")
    print(
        comparison
        .round(1)
        .to_string()
    )

    # Show which matching method was used
    if "resampling_method" in resampled.columns:

        method_summary = (
            resampled.groupby("resampling_method")
            .agg(
                rows=(ID_COL, "size"),
                unique_ids=(ID_COL, "nunique")
            )
            .reset_index()
        )

        print("\nResampling methods used:")
        print(method_summary.to_string(index=False))

    return resampled, comparison.reset_index()


# =============================================================================
# 8. Additional explicit train/test ID verification
# =============================================================================

def verify_no_ids_lost(original_df, resampled_df, label):
    """
    Compare original and resampled ID sets and print any discrepancies.
    """

    original_ids = set(
        original_df[ID_COL].astype(str).str.strip().unique()
    )

    resampled_ids = set(
        resampled_df[ID_COL].astype(str).str.strip().unique()
    )

    lost_ids = sorted(original_ids - resampled_ids)
    retained_ids = sorted(original_ids & resampled_ids)

    print(f"\n{'=' * 80}")
    print(f"{label}: FINAL ID VERIFICATION")
    print(f"{'=' * 80}")

    print(f"Original IDs: {len(original_ids)}")
    print(f"Retained IDs: {len(retained_ids)}")
    print(f"Lost IDs:     {len(lost_ids)}")

    if lost_ids:
        print(f"Lost ID list: {lost_ids}")

        raise AssertionError(
            f"{label} lost {len(lost_ids)} IDs during resampling."
        )

    print("Verification passed: no IDs were lost.")


# =============================================================================
# 8A: Define evaulate_six_hour_output: Checking completeness of the new six hour-data
# =============================================================================

def evaluate_six_hour_output(df, label):
    """
    Verify the sampling rate in the new six-hour dataset
    """

    check = df.sort_values(
        [ID_COL, DATE_COL]
    ).copy()

    check["interval_hours"] = (
        check.groupby(ID_COL)[DATE_COL]
        .diff()
        .dt.total_seconds()
        .div(3600)
    )

    check["calendar_date"] = check[DATE_COL].dt.date

    daily_counts = (
        check.groupby([ID_COL, "calendar_date"])
        .size()
        .rename("fixes_per_day")
        .reset_index()
    )

    valid_intervals = check.loc[
        check["interval_hours"] > 0,
        "interval_hours"
    ]

    complete_days = (
        daily_counts["fixes_per_day"] ==
        EXPECTED_FIXES_PER_DAY
    ).mean() * 100

    days_with_at_least_four = (
        daily_counts["fixes_per_day"] ==
        EXPECTED_FIXES_PER_DAY
    ).mean() * 100

    print(f"\n{'=' * 80}")
    print(f"{label}: VERIFICATION OF SIX-HOUR OUTPUT")
    print(f"{'=' * 80}")

    if not valid_intervals.empty:
        print(
            f"Median intervals: "
            f"{valid_intervals.median():2f} hours"
        )
        print(
            f"Mean interval: "
            f"{valid_intervals.mean():.2f} hours"
        )

        six_hour_percent = valid_intervals.between(
            5.99, 6.01
        ).mean() * 100

        print(
            f"Exactly six-hour intervals: "
            f"{six_hour_percent:.1f}%"
        )

        print("\nFixes per calander day:")
        print(
            daily_counts["fixes_per_day"]
            .describe()
            .to_string()
        )

        print(
            f"\nID-days containing excatly four fixes: "
            f"{complete_days:.1f}%"
        )

        print(
            f"ID-days containing at least our fixes "
            f"{days_with_at_least_four:.1f}%"
        )

        # Summary for each deer
        id_daily_summary = (
            daily_counts.groupby(ID_COL)["fixes_per_day"]
            .agg(
                observed_days="size",
                mean_fixes_per_day="mean",
                median_fixes_per_day="median",
                minimum_fixes_per_day="min",
                maximum_fixes_per_day="max"
            )
            .reset_index()
        )

        print("Per-ID daily fix summary after resampling:")
        print(
            id_daily_summary
            .round(2)
            .to_string(index=False)
        )

        return daily_counts, id_daily_summary


# =============================================================================
# 9. Main processing loop
# =============================================================================

all_original_data = {}
all_resampled_data = {}
all_retention_tables = {}

for label, file_name in DATASETS.items():

    input_path = BASE_PATH / file_name

    print(f"\n\n{'#' * 80}")
    print(f"PROCESSING {label.upper()} DATASET")
    print(f"Input: {input_path}")
    print(f"{'#' * 80}")

    if not input_path.exists():
        raise FileNotFoundError(
            f"File not found: {input_path}"
        )

    # -------------------------------------------------------------------------
    # Load and clean original data
    # -------------------------------------------------------------------------

    original_df = pd.read_csv(input_path)

    original_df = clean_datetime_data(
        original_df,
        label=f"{label} original"
    )

    print_unique_ids_and_status(
        original_df,
        label=f"{label} original"
    )

    original_eda = summarize_fix_rate(
        original_df,
        label=f"{label} original"
    )

    # -------------------------------------------------------------------------
    # Create six-hour dataset
    # -------------------------------------------------------------------------

    six_hour_df, retention_table = resample_dataset_to_six_hours(
        df=original_df,
        label=label,
        interval_hours=TARGET_INTERVAL_HOURS,
        tolerance_hours=MATCH_TOLERANCE_HOURS
    )

    # -------------------------------------------------------------------------
    # Verify six-hour output
    # -------------------------------------------------------------------------

    six_hour_eda = summarize_fix_rate(
        six_hour_df,
        label=f"{label} six-hour"
    )

    six_hour_daily_counts, six_hour_id_summary = (
        evaluate_six_hour_output(
            six_hour_df,
            label=label
        )
    )

    # -------------------------------------------------------------------------
    # Save output
    # -------------------------------------------------------------------------

    output_file_name = (
        Path(file_name).stem +
        "_6hour_resampled.csv"
    )

    output_path = OUTPUT_DIR / output_file_name

    six_hour_df.to_csv(
        output_path,
        index=False
    )

    retention_path = (
        OUTPUT_DIR /
        f"{label.lower()}_six_hour_retention_by_id.csv"
    )

    retention_table.to_csv(
        retention_path,
        index=False
    )

    daily_counts_path = (
        OUTPUT_DIR /
        f"{label.lower()}_six_hour_daily_fix_counts.csv"
    )

    six_hour_daily_counts.to_csv(
        daily_counts_path,
        index=False
    )

    id_summary_path = (
        OUTPUT_DIR /
        f"{label.lower()}_six_hour_id_summary.csv"
    )

    six_hour_id_summary.to_csv(
        id_summary_path,
        index=False
    )

    print(f"\nSaved six-hour dataset:")
    print(output_path)

    print("\nSaved supporting summaries:")
    print(retention_path)
    print(daily_counts_path)
    print(id_summary_path)

    all_original_data[label] = original_df
    all_resampled_data[label] = six_hour_df
    all_retention_tables[label] = retention_table


# =============================================================================
# 10. Combined before-versus-after summary
# =============================================================================

comparison_rows = []

for label in DATASETS:

    original_df = all_original_data[label]
    six_hour_df = all_resampled_data[label]

    original_daily = (
        original_df.assign(
            calendar_date=original_df[DATE_COL].dt.date
        )
        .groupby([ID_COL, "calendar_date"])
        .size()
    )

    six_hour_daily = (
        six_hour_df.assign(
            calendar_date=six_hour_df[DATE_COL].dt.date
        )
        .groupby([ID_COL, "calendar_date"])
        .size()
    )

    original_intervals = (
        original_df.sort_values([ID_COL, DATE_COL])
        .groupby(ID_COL)[DATE_COL]
        .diff()
        .dt.total_seconds()
        .div(3600)
    )

    six_hour_intervals = (
        six_hour_df.sort_values([ID_COL, DATE_COL])
        .groupby(ID_COL)[DATE_COL]
        .diff()
        .dt.total_seconds()
        .div(3600)
    )

    comparison_rows.append({
        "dataset": label,
        "version": "Original approximately 3-hour",
        "rows": len(original_df),
        "unique_ids": original_df[ID_COL].nunique(),
        "median_interval_hours": original_intervals.median(),
        "mean_fixes_per_id_day": original_daily.mean(),
        "median_fixes_per_id_day": original_daily.median()
    })

    comparison_rows.append({
        "dataset": label,
        "version": "Resampled 6-hour",
        "rows": len(six_hour_df),
        "unique_ids": six_hour_df[ID_COL].nunique(),
        "median_interval_hours": six_hour_intervals.median(),
        "mean_fixes_per_id_day": six_hour_daily.mean(),
        "median_fixes_per_id_day": six_hour_daily.median()
    })

comparison_summary = pd.DataFrame(comparison_rows)

comparison_output_path = (
    OUTPUT_DIR /
    "original_vs_six_hour_sampling_summary.csv"
)

comparison_summary.to_csv(
    comparison_output_path,
    index=False
)

print(f"\n\n{'=' * 80}")
print("FINAL ORIGINAL VERSUS SIX-HOUR COMPARISON")
print(f"{'=' * 80}")

print(
    comparison_summary
    .round(2)
    .to_string(index=False)
)

print("\nSaved combined comparison:")
print(comparison_output_path)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 100% experiment: leak-free preprocessing with ID-disjoint splits + train-only fit
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13)
np.random.seed(13)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base       = Path("<LOCAL_PATH>")

# inputs: training pool is CWD-free; test is CWD-positive
train_raw  = base / "CWD_negative_training_with_seasons2_6hour_resampled.csv"   # negatives only (still guard below)
test_raw   = base / "CWD_positive_testing_with_seasons2_6hour_resampled.csv"     # positives (mixed OK; we only transform)

# outputs
out_train  = base / "train_processed_100.csv"
out_val    = base / "val_processed_100.csv"
out_test   = base / "test_processed_100.csv"

enc_path   = base / "encoder_100.pkl"
sca_path   = base / "scaler_100.pkl"
feat_path  = base / "feature_columns_100.pkl"
split_path = base / "split_ids_100.pkl"

to_drop = [
    'date_std', 'dx','dy','dt','burst','seg_key'
]

num_feats = [
    'velo','abs_rel_angle','sinuosity',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'akde_low','akde_est','akde_high'
]

cat_feats  = ['season']
meta_feats = ['id','date']

# ── UTILITIES ──────────────────────────────────────────────────────────────────
def basic_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unused cols, enforce numeric types, keep pkey intact, preserve order."""
    df = df.drop(columns=to_drop, errors='ignore').copy()
    
    # Preserve original datetime string AND a parsed version
    df['datetime_raw'] = df['date']   # keep raw untouched
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    for c in num_feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    df = df.dropna(subset=num_feats + cat_feats).copy()
    
    # Sort by id + parsed datetime (if valid), fallback to pkey for stability
    df = df.sort_values(['id', 'date']).reset_index(drop=True)
    return df

def transform_with_artifacts(df, encoder, scaler, feature_columns):
    """Apply fitted encoder/scaler; return DataFrame with pkey + meta + features."""
    cat_mat = encoder.transform(df[cat_feats])
    num_mat = scaler.transform(df[num_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()
    X = np.hstack([num_mat, cat_mat])
    feat_df = pd.DataFrame(X, columns=num_feats + cat_names)

    feat_df = feat_df.reindex(columns=feature_columns, fill_value=0.0)
    out = pd.concat([df[meta_feats].reset_index(drop=True),
                     feat_df.reset_index(drop=True)], axis=1)
    return out

# ── 1) LOAD TRAIN POOL ────────────────────────────────────────────────────────
df_train_pool = pd.read_csv(train_raw)
df_train_pool = df_train_pool[df_train_pool['cwd_status'] == "negative"].copy()
df_train_pool = basic_cleanup(df_train_pool)

# ── 2) ID-DISJOINT SPLIT ──────────────────────────────────────────────────────
ids = df_train_pool['id'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=13)
train_idx, val_idx = next(gss.split(df_train_pool, groups=df_train_pool['id']))

train_ids = df_train_pool.iloc[train_idx]['id'].unique()
val_ids   = df_train_pool.iloc[val_idx]['id'].unique()

df_train = df_train_pool[df_train_pool['id'].isin(train_ids)].copy()
df_val   = df_train_pool[df_train_pool['id'].isin(val_ids)].copy()

# ── 3) FIT ENCODER & SCALER ON TRAIN ONLY ─────────────────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler  = MinMaxScaler()

encoder.fit(df_train[cat_feats])
scaler.fit(df_train[num_feats])

cat_names = encoder.get_feature_names_out(cat_feats).tolist()
feature_columns = num_feats + cat_names

# ── 4) TRANSFORM SPLITS ──────────────────────────────────────────────────────
df_train_out = transform_with_artifacts(df_train, encoder, scaler, feature_columns)
df_val_out   = transform_with_artifacts(df_val,   encoder, scaler, feature_columns)

# ── 5) TEST ───────────────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_raw = basic_cleanup(df_test_raw)
df_test_out = transform_with_artifacts(df_test_raw, encoder, scaler, feature_columns)

# ── 6) SAVE ARTIFACTS ─────────────────────────────────────────────────────────
joblib.dump(encoder, enc_path)
joblib.dump(scaler,  sca_path)
joblib.dump(feature_columns, feat_path)

# always keep IDs as strings
train_ids_str = sorted({str(x).strip() for x in train_ids})
val_ids_str   = sorted({str(x).strip() for x in val_ids})
test_ids_str  = sorted({str(x).strip() for x in df_test_out['id'].astype(str).unique()})

joblib.dump(
    {"train_ids": train_ids_str,
     "val_ids":   val_ids_str,
     "test_ids":  test_ids_str},
    split_path
)

df_train_out.to_csv(out_train, index=False)
df_val_out.to_csv(out_val, index=False)
df_test_out.to_csv(out_test, index=False)

print("✅ Preprocessing complete with pkey preserved.")
print("Train IDs:", train_ids_str)
print("Val IDs:",   val_ids_str)
print("Test IDs:",  test_ids_str)




In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Autoencoder training USING CUSTOM LOSS + flexible thresholding
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Config / Paths =====
base = "<LOCAL_PATH>"
train_path = f"{base}/train_processed_100.csv"
val_path   = f"{base}/val_processed_100.csv"   # negatives-only val (from preprocessing)
test_path  = f"{base}/test_processed_100.csv"

feat_cols_path = f"{base}/feature_columns_100.pkl"
model_path     = f"{base}/autoencoder_model.h5"
threshold_path = f"{base}/reconstruction_threshold.pkl"
history_path   = f"{base}/training_history.npy"
ckpt_path      = f"{base}/ae_ckpt.weights.h5"

# ===== Threshold policy (choose one) =====
THRESHOLD_POLICY = "train_lowest_p95"   # "train_lowest_p95" | "train_quantile" | "val_row_fpr" | "val_id_flag_rate" | "mad"

# Params for each policy
LOWEST_FRAC   = 0.75      # for "train_lowest_p95" 
LOWEST_Q      = 90        # percentile on the lowest fraction
TRAIN_Q       = 0.90      # for "train_quantile": e.g., 90th percentile of all train errors
VAL_ROW_FPR   = 0.20      # for "val_row_fpr": allow 20% of NEG val rows to flag → lower θ, more flags
VAL_ID_RATE   = 0.20      # for "val_id_flag_rate": allow 20% of NEG val IDs to flag at least 1 row
MIN_ROWS_PER_ID = 1       # ignore super-short IDs when computing ID-level rate
MAD_K         = 3.0       # for "mad"



# ===== Load features with frozen column order =====
feature_columns = joblib.load(feat_cols_path)

def load_X(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    X = df[feature_columns].values.astype(np.float32)
    meta = df[["id"]].copy()
    if "datetime" in df.columns:
        meta["datetime"] = df["datetime"].values
    return X, meta, df

X_train, meta_train, df_train_raw = load_X(train_path)
X_val,   meta_val,   df_val_raw   = load_X(val_path)
X_test,  meta_test,  df_test_raw  = load_X(test_path)

# ===== tf.data pipelines =====
batch_size = 180
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, X))
    if training:
        ds = ds.shuffle(min(10000, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, training=True)
val_ds   = make_ds(X_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(1e-4)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(inp)
x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
z = layers.Dense(4,  activation='relu', kernel_regularizer=reg, name="bottleneck")(x)
y = layers.Dense(16, activation='relu', kernel_regularizer=reg)(z)
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
out = layers.Dense(input_dim, activation='linear')(y)
autoencoder = Model(inp, out, name="autoencoder")

# === Custom loss ===
LAMBDA_DEV = 0.5                      # ← your choice
LOSS_SCALE = 1.0 + LAMBDA_DEV*input_dim  # α = 1 + λ d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=ckpt_path, save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop, ckpt], verbose=1
)

autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Saved model → {model_path}")

# ===== Errors on train/val (row-level, using SAME loss) =====
recon_train = autoencoder.predict(X_train, batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict(X_val, batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

# ===== Threshold policies =====
def choose_threshold(policy: str):
    info = {"policy": policy}   # <— standard key

    if policy == "train_lowest_p95":
        n_low = int(len(train_errors) * LOWEST_FRAC)
        base = np.sort(train_errors)[:n_low]
        theta = float(np.percentile(base, LOWEST_Q))
        info.update({"lowest_frac": LOWEST_FRAC, "percentile": LOWEST_Q, "n_low": n_low})

    elif policy == "train_quantile":
        theta = float(np.quantile(train_errors, TRAIN_Q))
        info.update({"train_quantile": TRAIN_Q})

    elif policy == "val_row_fpr":
        q = 1.0 - float(VAL_ROW_FPR)
        theta = float(np.quantile(val_errors, q))
        info.update({"val_row_fpr": VAL_ROW_FPR, "val_quantile": q})

    elif policy == "val_id_flag_rate":
        q_grid = np.linspace(0.50, 0.999, 200)
        ids = df_val_raw["id"].values
        best_theta, best_rate, best_q = None, None, None
        for q in q_grid:
            th = np.quantile(val_errors, q)
            row_flag = val_errors > th
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            rate = float(id_any.reindex(eligible).fillna(False).mean())
            if rate <= VAL_ID_RATE:
                best_theta, best_rate, best_q = float(th), rate, float(q)
                break
        if best_theta is None:
            best_q = float(q_grid[-1])
            best_theta = float(np.quantile(val_errors, best_q))
            row_flag = val_errors > best_theta
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            best_rate = float(id_any.reindex(eligible).fillna(False).mean())
        theta = best_theta
        info.update({"val_id_rate_target": VAL_ID_RATE, "achieved_rate": best_rate, "val_quantile": best_q})

    elif policy == "mad":
        med = float(np.median(train_errors))
        mad = float(np.median(np.abs(train_errors - med)) + 1e-12)
        theta = med + MAD_K * 1.4826 * mad
        info.update({"median": med, "mad": mad, "k": MAD_K})

    else:
        raise ValueError(f"Unknown THRESHOLD_POLICY: {policy}")

    return theta, info

theta, th_info = choose_threshold(THRESHOLD_POLICY)

# Persist policy hyperparameters for reproducibility
if THRESHOLD_POLICY == "train_lowest_p95":
    th_info.update({"LOWEST_FRAC": float(LOWEST_FRAC), "LOWEST_Q": float(LOWEST_Q)})
elif THRESHOLD_POLICY == "train_quantile":
    th_info.update({"TRAIN_Q": float(TRAIN_Q)})
elif THRESHOLD_POLICY == "val_row_fpr":
    th_info.update({"VAL_ROW_FPR": float(VAL_ROW_FPR)})
elif THRESHOLD_POLICY == "val_id_flag_rate":
    th_info.update({"VAL_ID_RATE": float(VAL_ID_RATE), "MIN_ROWS_PER_ID": int(MIN_ROWS_PER_ID)})
elif THRESHOLD_POLICY == "mad":
    th_info.update({"MAD_K": float(MAD_K)})

th_info["effective_multiplier"] = float(LOSS_SCALE)  # alias for reviewers
print(f"[loss] λ={LAMBDA_DEV} | d={input_dim} | α=1+λ·d={LOSS_SCALE:.3f} (scaled-MSE)")


# ===== Diagnostics: how aggressive is θ? =====
def percent_rows_flagged(errors, theta):
    return float(np.mean(errors > theta))

train_row_rate = percent_rows_flagged(train_errors, theta)
val_row_rate   = percent_rows_flagged(val_errors, theta)

# ID-level on val (negatives)
val_ids = df_val_raw["id"].values
val_row_flag = val_errors > theta
val_id_any = pd.Series(val_row_flag).groupby(val_ids, sort=False).any()
val_n_rows = pd.Series(1, index=val_ids).groupby(val_ids).sum()
val_eligible = val_n_rows[val_n_rows >= MIN_ROWS_PER_ID].index
val_id_rate = float(val_id_any.reindex(val_eligible).fillna(False).mean())

print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")
print(f"   Train rows flagged: {100*train_row_rate:.1f}%")
print(f"   Val   rows flagged: {100*val_row_rate:.1f}%")
print(f"   Val     IDs flagged (≥1 row): {100*val_id_rate:.1f}% (min_rows={MIN_ROWS_PER_ID})")

# ===== Persist threshold + bookkeeping =====
# Diagnostics
# (unchanged, but print 'policy')
print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")

# Persist threshold + bookkeeping (use 'policy')
th_info.update({
    "policy": THRESHOLD_POLICY,
    "theta": float(theta),
    "lambda_dev": float(LAMBDA_DEV),
    "input_dim": int(input_dim),
    "loss_scale": float(LOSS_SCALE),  # α
    "loss_name": "scaled_mse_equivalent_to_mse_plus_lambda_sse",
    "train_rows_flagged": train_row_rate,
    "val_rows_flagged": val_row_rate,
    "val_ids_flagged": val_id_rate,
    "min_rows_per_id": int(MIN_ROWS_PER_ID)
})
joblib.dump(th_info, threshold_path)
print(f"✅ Threshold saved → {threshold_path}")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
threshold_path  = base / "reconstruction_threshold.pkl"   # contains θ + loss params from training
feat_cols_path  = base / "feature_columns_100.pkl"      # enforce column order
param_out_path  = base / "id_rule_params.pkl"

USE_MIXED_VAL = False  # your val is negatives-only per preprocessing

# preregistered operating point (choose & freeze before test)
FROZEN_COUNT     = 110
FROZEN_FRAC      = 0.75
MIN_ROWS_PER_ID  = 1          # consider 100 if short IDs are noisy
AGGREGATION_RULE = "OR"       # freeze: "OR", "AND", "FRAC_ONLY", or "COUNT_ONLY"

# if you ever use mixed val with labels, these grids are considered a priori
COUNTS = [50, 100, 150, 200]
FRACS  = [0.80, 0.90, 0.925]
POS_IDS_VAL = []   # only used when USE_MIXED_VAL=True

# ── LOAD MODEL / FEATURES / TRAIN ARTIFACT ────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)
autoencoder = load_model(model_path, compile=False)

# load training artifact with θ + loss params
th_info   = joblib.load(threshold_path)
theta     = float(th_info["theta"])
policy    = th_info.get("policy", "unknown")

# reconstruct scaled-MSE loss consistently
lambda_dev = float(th_info.get("lambda_dev", 0.5))
input_dim  = int(th_info.get("input_dim", len(feature_columns)))
loss_scale = float(th_info.get("loss_scale", 1.0 + lambda_dev * input_dim))

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
print(f"Using TRAIN-derived θ = {theta:.6f} (policy={policy}) | λ={lambda_dev} | d={input_dim} | α={loss_scale:.6g}")

# ── LOAD VAL AND COMPUTE ROW ERRORS (FROZEN FEATURE ORDER) ────────────────────
df_val = pd.read_csv(val_csv)
if "datetime" in df_val.columns:
    df_val["datetime"] = pd.to_datetime(df_val["datetime"], errors="coerce")

X_val     = df_val[feature_columns].values.astype(np.float32)
recon_val = autoencoder.predict(X_val, batch_size=128, verbose=0)
val_err   = loss_fn(X_val, recon_val).numpy().astype(np.float64)

rows_val = pd.DataFrame({
    "id": df_val["id"].values,
    "datetime": df_val["datetime"] if "datetime" in df_val.columns else pd.NaT,
    "reconstruction_error": val_err
})
rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

# ── AGGREGATE BY ID ───────────────────────────────────────────────────────────
g         = rows_val.groupby("id", sort=False)
n_rows    = g.size()
n_flags   = g["row_flag"].sum()
flag_frac = n_flags / n_rows

# consistent eligible set (evaluation/selection must use the same gate)
eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_rows    = n_rows.reindex(eligible_idx).astype(int)
n_flags   = n_flags.reindex(eligible_idx).astype(int)
flag_frac = flag_frac.reindex(eligible_idx).astype(float)

# ── SELECT (COUNT, FRAC) IF MIXED VAL; OTHERWISE FREEZE A PRIORI ─────────────
if USE_MIXED_VAL:
    pos_val = set(POS_IDS_VAL) & set(eligible_idx)
    if len(pos_val) == 0:
        raise ValueError("No positive IDs present in eligible validation set.")

    best = None; best_key = None
    for c in COUNTS:
        for f in FRACS:
            idx_count = set(n_flags[n_flags >= c].index)
            idx_frac  = set(flag_frac[flag_frac >= f].index)

            if   AGGREGATION_RULE == "AND":       pred = idx_count & idx_frac
            elif AGGREGATION_RULE == "FRAC_ONLY": pred = idx_frac
            elif AGGREGATION_RULE == "COUNT_ONLY":pred = idx_count
            else:                                 pred = idx_count | idx_frac  # OR

            recall = len(pred & pos_val) / len(pos_val)
            key = (recall, c, f)  # maximize recall; tie-break to stricter (higher c,f)
            if (best is None) or (key > best_key):
                best = {"count": int(c), "frac": float(f), "val_recall": float(recall)}
                best_key = key

    COUNT, FRAC = best["count"], best["frac"]
    print(f"Frozen ID rule from validation (positives-only, {AGGREGATION_RULE}): "
          f"COUNT={COUNT} | FRAC={FRAC} | recall={best['val_recall']:.3f}")
else:
    COUNT, FRAC = int(FROZEN_COUNT), float(FROZEN_FRAC)
    print(f"Using pre-registered ID rule ({AGGREGATION_RULE}; negatives-only val): "
          f"COUNT={COUNT} | FRAC={FRAC} | min_rows={MIN_ROWS_PER_ID}")

# ── SAVE FROZEN PARAMS FOR TEST ───────────────────────────────────────────────
frozen = {
    "theta": float(theta),
    "count": int(COUNT),
    "frac": float(FRAC),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(input_dim),
    "loss_scale": float(loss_scale),
    "policy": policy,
    "aggregation_rule": AGGREGATION_RULE,  # ← frozen & explicit
    "val_meta": {
        "min_rows_per_id": int(MIN_ROWS_PER_ID),
        "use_mixed_val": bool(USE_MIXED_VAL)
    }
}
joblib.dump(frozen, param_out_path)
print(f"✅ Saved frozen params to: {param_out_path}")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
test_feat_path  = base / "test_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
frozen_param_p  = base / "id_rule_params.pkl"           # produced in validation (frozen params)
feat_cols_path  = base / "feature_columns_100.pkl"      # frozen feature order

rows_out_csv    = str(test_feat_path).replace(".csv", "_rows_with_flags_FROZEN.csv")
ids_out_csv     = str(test_feat_path).replace(".csv", "_id_summary_FROZEN.csv")
residuals_out   = str(test_feat_path).replace(".csv", "_rows_with_recon_and_residuals.csv")

# (Optional) test labels for evaluation
actual_cwd_instances_test     = ['A41', 'A43', 'A52', 'A56', 'A60', 'F106', 'F107', 'F26', 'F27', 'F30', 'F48', 'F62', 'F66', 'F73', 'F78', 'F79', 'F83', 'F87', 'F94']
unaffected_cwd_instances_test = []

# ── LOAD ARTIFACTS ────────────────────────────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)           # list[str]
frozen = joblib.load(frozen_param_p)                    # dict from validation

theta  = float(frozen["theta"])
COUNT  = int(frozen["count"])
FRAC   = float(frozen["frac"])
policy = frozen.get("policy", "unknown")

# loss consistency (scaled-MSE, α = 1 + λ·d)
lambda_dev = float(frozen.get("lambda_dev", 0.5))
input_dim  = int(frozen.get("input_dim", len(feature_columns)))
loss_scale = float(frozen.get("loss_scale", 1.0 + lambda_dev * input_dim))

AGGREGATION_RULE = str(frozen.get("aggregation_rule", "OR")).upper()
MIN_ROWS_PER_ID  = int(frozen.get("val_meta", {}).get("min_rows_per_id", 1))

print(f"Using frozen params → θ={theta:.6f} | COUNT={COUNT} | FRAC={FRAC} | "
      f"λ={lambda_dev} | d={input_dim} | α={loss_scale:.3f} | MIN_ROWS={MIN_ROWS_PER_ID} | "
      f"policy={policy} | agg_rule={AGGREGATION_RULE}")

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
autoencoder = load_model(model_path, compile=False)

# ── LOAD TEST + COMPUTE RECONS + RESIDUALS ────────────────────────────────────
df = pd.read_csv(test_feat_path)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

X_test = df[feature_columns].values.astype(np.float32)
recon  = autoencoder.predict(X_test, batch_size=128, verbose=0)

row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta

# absolute residuals per feature
residuals = np.abs(X_test - recon)
df_resid = pd.DataFrame(residuals, columns=[f"{c}_resid" for c in feature_columns])
df_recon = pd.DataFrame(recon, columns=[f"{c}_recon" for c in feature_columns])

rows_full = pd.concat([
    df.reset_index(drop=True)[["id", "date"]],
    pd.DataFrame({"reconstruction_error": row_err, "row_flag": row_flag}),
    df_recon, df_resid
], axis=1)

rows_full.to_csv(residuals_out, index=False)
print(f"✅ Saved per-row reconstructions + residuals → {residuals_out}")

# ── CONTINUE WITH AGGREGATION / IDS SUMMARY (unchanged) ───────────────────────
g        = pd.DataFrame({"row_flag": row_flag, "err": row_err, "id": df["id"]}).groupby("id", sort=False)
n_rows   = g.size()
n_flags  = g["row_flag"].sum()
flag_fr  = n_flags / n_rows
id_max   = g["err"].max()
id_mean  = g["err"].mean()

eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_flags_elig = n_flags.reindex(eligible_idx)
flag_fr_elig = flag_fr.reindex(eligible_idx)

if AGGREGATION_RULE == "AND":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "FRAC_ONLY":
    pred_ids = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "COUNT_ONLY":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index)
else:  # OR
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)

pred_ids = sorted(pred_ids)
print("Predicted positive IDs:", pred_ids)

# =============================================================================
# SAVE EXACT TEST DATA + MODEL FLAGS FOR FEATURE-CONTRIBUTION ANALYSIS
# =============================================================================
#
# This MUST occur here, while:
#
#   df        = exact test dataframe used to construct X_test
#   X_test    = exact feature matrix supplied to this autoencoder
#   recon     = reconstructions from this autoencoder
#   row_err   = reconstruction errors from this autoencoder
#   row_flag  = row-level anomaly classifications from this autoencoder
#   pred_ids  = ID-level predictions from the frozen decision rule
#
# are all still aligned.
# =============================================================================

if not (
    len(df)
    == len(X_test)
    == len(recon)
    == len(row_err)
    == len(row_flag)
):

    raise ValueError(
        "\nFEATURE-CONTRIBUTION ALIGNMENT FAILURE\n"
        f"df rows:       {len(df):,}\n"
        f"X_test rows:   {len(X_test):,}\n"
        f"recon rows:    {len(recon):,}\n"
        f"row_err rows:  {len(row_err):,}\n"
        f"row_flag rows: {len(row_flag):,}"
    )


# -------------------------------------------------------------------------
# Verify that df still contains the exact matrix used for prediction
# -------------------------------------------------------------------------

X_test_check = (
    df[feature_columns]
    .to_numpy(dtype=np.float32)
)

if X_test_check.shape != X_test.shape:

    raise ValueError(
        "\nFEATURE MATRIX SHAPE MISMATCH\n"
        f"df-derived matrix: {X_test_check.shape}\n"
        f"X_test:            {X_test.shape}"
    )


if not np.allclose(
    X_test_check,
    X_test,
    rtol=1e-6,
    atol=1e-7,
    equal_nan=True,
):

    raise ValueError(
        "\nFEATURE MATRIX CONTENT MISMATCH\n"
        "The current df is no longer the dataframe used to create X_test."
    )


# -------------------------------------------------------------------------
# Add model outputs to the exact processed test dataframe
# -------------------------------------------------------------------------

df_test_for_contrib = (
    df
    .reset_index(drop=True)
    .copy()
)

df_test_for_contrib["row_flag"] = (
    row_flag.astype(bool)
)

df_test_for_contrib["reconstruction_error"] = (
    row_err.astype(float)
)


# -------------------------------------------------------------------------
# Save contribution-ready test dataframe
# -------------------------------------------------------------------------

PROC_TEST_WITH_FLAGS = str(
    test_feat_path
).replace(
    ".csv",
    "_WITH_FLAGS.csv"
)

df_test_for_contrib.to_csv(
    PROC_TEST_WITH_FLAGS,
    index=False
)


print("\n" + "=" * 80)
print("FEATURE-CONTRIBUTION INPUT CREATED")
print("=" * 80)

print(
    f"Rows:                 "
    f"{len(df_test_for_contrib):,}"
)

print(
    f"Anomalous rows:       "
    f"{row_flag.sum():,}"
)

print(
    f"Anomalous rows (%):   "
    f"{100 * row_flag.mean():.2f}%"
)

print(
    f"Predicted CWD+ IDs:   "
    f"{len(pred_ids)}"
)

print(
    "\nSaved:"
)

print(PROC_TEST_WITH_FLAGS)

rows = pd.DataFrame({
    "id": df["id"].values,
    "date": df["date"] if "date" in df.columns else pd.NaT,
    "reconstruction_error": row_err,
    "row_flag": row_flag
})
rows.to_csv(rows_out_csv, index=False)


# Rule-aligned and component scores (for plots & AUC)
eps = 1e-12
count_norm_all = n_flags / max(COUNT, eps)
frac_norm_all  = flag_fr / max(FRAC, eps)

if AGGREGATION_RULE == "AND":
    score_all = np.minimum(count_norm_all, frac_norm_all)
elif AGGREGATION_RULE == "FRAC_ONLY":
    score_all = frac_norm_all
elif AGGREGATION_RULE == "COUNT_ONLY":
    score_all = count_norm_all
else:  # OR
    score_all = np.maximum(count_norm_all, frac_norm_all)

ids_df = pd.DataFrame({
    "id": n_rows.index,
    "n_rows": n_rows.values,
    "n_flags": n_flags.reindex(n_rows.index).values,
    "flag_frac": flag_fr.reindex(n_rows.index).values,
    "rule_aligned_score": score_all.reindex(n_rows.index).values,
    "frac_norm_score": frac_norm_all.reindex(n_rows.index).values,
    "count_norm_score": count_norm_all.reindex(n_rows.index).values,
    "id_max_error": id_max.reindex(n_rows.index).values,
    "id_mean_error": id_mean.reindex(n_rows.index).values,
    "eligible": [1 if i in set(eligible_idx) else 0 for i in n_rows.index],
    "is_positive_id": [1 if i in pred_ids else 0 for i in n_rows.index]
})
ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ Wrote: {rows_out_csv}\n✅ Wrote: {ids_out_csv}")

# ── (OPTIONAL) TEST EVALUATION — eligible set only, rule-aligned score ────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)

eval_ids   = sorted((pos_set | neg_set) & set(eligible_idx))
n_pos_eval = sum(i in pos_set for i in eval_ids)
n_neg_eval = sum(i in neg_set for i in eval_ids)
print(f"Eval on eligible IDs only → n_eval={len(eval_ids)} (pos={n_pos_eval}, neg={n_neg_eval}; min_rows={MIN_ROWS_PER_ID})")

if eval_ids and (n_pos_eval > 0 or n_neg_eval > 0):
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    if (y_true == 1).all():
        TP = np.sum(y_pred == 1)
        FN = np.sum(y_pred == 0)
        FP = 0
        TN = 0

        sensitivity = TP / (TP + FN) if (TP + FN) else 0.0
        precision   = TP / (TP + FP) if (TP + FP) else 0.0
        accuracy    = (TP + TN) / (TP + FN + TN + FP) if (TP + FN + TN + FP) else 0.0
        f1_score    = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0.0
        prevalence  = 1.0
        tss         = np.nan

        print(f"TP = {TP}, FN = {FN}")
        print(f"Sensitivity     (TPR) = {sensitivity:.3f}")
        print(f"Precision              = {precision:.3f}")
        print(f"Accuracy               = {accuracy:.3f}")
        print(f"F1 Score               = {f1_score:.3f}")
        print(f"Prevalence Identified  = {prevalence:.3f}")
        print(f"TSS                   = N/A (no negatives)")


# Comparison of thresholding between Mule Deer and White-Tailed Deer

In [ ]:
# =============================================================================
# FAST LEARNING-RATE COMPARISON
# White-Tailed Deer vs. Mule Deer Autoencoders
#
# Stage 1:
#   - 2 datasets
#   - 4 learning rates
#   - 1 seed
#   - Maximum 30 epochs
#   - Representative row samples
#
# This is intended to identify which learning rates are clearly too slow,
# unstable, or poorly performing. Afterward, rerun only the best two rates
# with more seeds and the complete datasets.
# =============================================================================

import os
import gc
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Model, layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping


# =============================================================================
# CONFIGURATION
# =============================================================================

SEED = 13

BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 5

LAMBDA_DEV = 0.5
L2_REGULARIZATION = 1e-4

LEARNING_RATES = [
    1e-6,
    1e-4,
]

# Representative samples for the screening analysis.
# Increase later only for the final confirmation.
MAX_TRAIN_ROWS = 100_000
MAX_VAL_ROWS = 30_000

OUTPUT_DIR = Path(
    "<LOCAL_PATH>"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


DATASETS = {
    "White_Tailed_Deer": {
        "base": Path("<LOCAL_PATH>"),
        "train_file": "train_processed_100.csv",
        "val_file": "val_processed_100.csv",
        "feature_file": "feature_columns_100.pkl",
        "original_learning_rate": 1e-6,
    },

    "Mule_Deer": {
        "base": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment"
        ),
        "train_file": "train_processed_60_40.csv",
        "val_file": "val_processed_60_40.csv",
        "feature_file": "feature_columns_60_40.pkl",
        "original_learning_rate": 1e-4,
    },
}


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# =============================================================================
# LOAD AND SAMPLE DATA
# =============================================================================

def sample_rows(X, max_rows, seed):
    """
    Randomly sample rows without replacement.

    The same sampled rows are reused across all learning rates for a dataset,
    ensuring a fair comparison.
    """
    if max_rows is None or len(X) <= max_rows:
        return X

    rng = np.random.default_rng(seed)

    indices = rng.choice(
        len(X),
        size=max_rows,
        replace=False,
    )

    return X[indices]


def load_dataset(dataset_name, config):
    """
    Load training and negative-only validation data using the frozen feature
    order saved for each analysis.
    """
    base = config["base"]

    train_path = base / config["train_file"]
    val_path = base / config["val_file"]
    feature_path = base / config["feature_file"]

    for path in [train_path, val_path, feature_path]:
        if not path.exists():
            raise FileNotFoundError(
                f"Missing file for {dataset_name}:\n{path}"
            )

    feature_columns = list(
        joblib.load(feature_path)
    )

    train_df = pd.read_csv(
        train_path,
        usecols=feature_columns,
    )

    val_df = pd.read_csv(
        val_path,
        usecols=feature_columns,
    )

    X_train = train_df.to_numpy(
        dtype=np.float32
    )

    X_val = val_df.to_numpy(
        dtype=np.float32
    )

    if not np.isfinite(X_train).all():
        raise ValueError(
            f"{dataset_name} training data contains NaN or infinity."
        )

    if not np.isfinite(X_val).all():
        raise ValueError(
            f"{dataset_name} validation data contains NaN or infinity."
        )

    X_train = sample_rows(
        X_train,
        max_rows=MAX_TRAIN_ROWS,
        seed=SEED,
    )

    X_val = sample_rows(
        X_val,
        max_rows=MAX_VAL_ROWS,
        seed=SEED + 1,
    )

    return {
        "dataset": dataset_name,
        "feature_columns": feature_columns,
        "X_train": X_train,
        "X_val": X_val,
        "original_learning_rate":
            config["original_learning_rate"],
    }


# =============================================================================
# MODEL
# =============================================================================

def build_autoencoder(input_dim):
    regularizer = regularizers.l2(
        L2_REGULARIZATION
    )

    inputs = layers.Input(
        shape=(input_dim,)
    )

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizer,
    )(inputs)

    x = layers.Dense(
        16,
        activation="relu",
        kernel_regularizer=regularizer,
    )(x)

    x = layers.Dense(
        4,
        activation="relu",
        kernel_regularizer=regularizer,
        name="bottleneck",
    )(x)

    x = layers.Dense(
        16,
        activation="relu",
        kernel_regularizer=regularizer,
    )(x)

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizer,
    )(x)

    outputs = layers.Dense(
        input_dim,
        activation="linear",
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="autoencoder",
    )


def make_scaled_loss(input_dim):
    """
    Reproduce the scaled MSE from the original analyses.
    """
    loss_scale = (
        1.0
        + LAMBDA_DEV * input_dim
    )

    def scaled_mse(y_true, y_pred):
        row_mse = tf.reduce_mean(
            tf.square(y_true - y_pred),
            axis=-1,
        )

        return loss_scale * row_mse

    return scaled_mse, float(loss_scale)


# =============================================================================
# INITIAL GRADIENT DIAGNOSTIC
# =============================================================================

def calculate_initial_gradient_norm(
    X_train,
    input_dim,
    learning_rate,
):
    """
    Calculate initial gradient magnitude using one batch.

    This helps determine whether the same nominal learning rate creates a much
    larger effective update for one dataset.
    """
    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(SEED)

    model = build_autoencoder(input_dim)
    loss_fn, _ = make_scaled_loss(input_dim)

    batch_size = min(
        BATCH_SIZE,
        len(X_train),
    )

    batch = tf.convert_to_tensor(
        X_train[:batch_size],
        dtype=tf.float32,
    )

    with tf.GradientTape() as tape:
        reconstruction = model(
            batch,
            training=True,
        )

        reconstruction_loss = tf.reduce_mean(
            loss_fn(
                batch,
                reconstruction,
            )
        )

        if model.losses:
            regularization_loss = tf.add_n(
                model.losses
            )
        else:
            regularization_loss = tf.constant(
                0.0,
                dtype=tf.float32,
            )

        total_loss = (
            reconstruction_loss
            + regularization_loss
        )

    gradients = tape.gradient(
        total_loss,
        model.trainable_variables,
    )

    gradients = [
        gradient
        for gradient in gradients
        if gradient is not None
    ]

    gradient_norm = float(
        tf.linalg.global_norm(
            gradients
        ).numpy()
    )

    weight_norm = float(
        tf.linalg.global_norm(
            model.trainable_variables
        ).numpy()
    )

    approximate_relative_update = (
        learning_rate
        * gradient_norm
        / max(weight_norm, 1e-12)
    )

    return {
        "initial_total_loss":
            float(total_loss.numpy()),
        "initial_gradient_norm":
            gradient_norm,
        "initial_weight_norm":
            weight_norm,
        "approximate_relative_update":
            approximate_relative_update,
    }


# =============================================================================
# TRAIN ONE MODEL
# =============================================================================

def run_learning_rate(
    dataset,
    learning_rate,
):
    """
    Train one model and calculate validation reconstruction performance.
    """
    dataset_name = dataset["dataset"]

    X_train = dataset["X_train"]
    X_val = dataset["X_val"]

    input_dim = X_train.shape[1]

    gradient_metrics = (
        calculate_initial_gradient_norm(
            X_train=X_train,
            input_dim=input_dim,
            learning_rate=learning_rate,
        )
    )

    # Clear the model used for gradient diagnostics.
    tf.keras.backend.clear_session()
    gc.collect()

    # Reset the seed so every learning rate begins from the same weights.
    tf.keras.utils.set_random_seed(SEED)

    model = build_autoencoder(input_dim)

    loss_fn, loss_scale = make_scaled_loss(
        input_dim
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=loss_fn,
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    print(
        f"\nTraining {dataset_name} "
        f"with learning rate {learning_rate:.0e}"
    )

    history = model.fit(
        X_train,
        X_train,
        validation_data=(X_val, X_val),
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        shuffle=True,
        callbacks=[early_stopping],
        verbose=1,
    )

    train_loss_history = np.asarray(
        history.history["loss"],
        dtype=np.float64,
    )

    val_loss_history = np.asarray(
        history.history["val_loss"],
        dtype=np.float64,
    )

    epochs_completed = len(
        train_loss_history
    )

    best_epoch = int(
        np.argmin(val_loss_history) + 1
    )

    best_scaled_val_loss = float(
        np.min(val_loss_history)
    )

    # Exact reconstruction error is calculated only once after training.
    val_reconstruction = model.predict(
        X_val,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    val_row_mse = np.mean(
        np.square(
            X_val.astype(np.float64)
            - val_reconstruction.astype(np.float64)
        ),
        axis=1,
    )

    exact_val_unscaled_mse = float(
        np.mean(val_row_mse)
    )

    exact_val_median_mse = float(
        np.median(val_row_mse)
    )

    exact_val_p95_mse = float(
        np.quantile(val_row_mse, 0.95)
    )

    result = {
        "dataset": dataset_name,
        "learning_rate": learning_rate,
        "original_learning_rate":
            dataset["original_learning_rate"],
        "is_original_learning_rate":
            learning_rate
            == dataset["original_learning_rate"],
        "n_train_rows": len(X_train),
        "n_validation_rows": len(X_val),
        "n_features": input_dim,
        "loss_scale": loss_scale,
        "batch_size": BATCH_SIZE,
        "epochs_completed": epochs_completed,
        "best_epoch": best_epoch,
        "best_scaled_validation_loss":
            best_scaled_val_loss,
        "exact_validation_unscaled_mse":
            exact_val_unscaled_mse,
        "exact_validation_median_row_mse":
            exact_val_median_mse,
        "exact_validation_p95_row_mse":
            exact_val_p95_mse,
        **gradient_metrics,
    }

    history_df = pd.DataFrame({
        "epoch": np.arange(
            1,
            epochs_completed + 1,
        ),
        "training_scaled_loss":
            train_loss_history,
        "validation_scaled_loss":
            val_loss_history,
        "dataset": dataset_name,
        "learning_rate": learning_rate,
    })

    tf.keras.backend.clear_session()
    gc.collect()

    return result, history_df


# =============================================================================
# PLOT RESULTS
# =============================================================================

def plot_validation_performance(
    results_df,
    output_path,
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    for dataset_name, group in results_df.groupby(
        "dataset"
    ):
        group = group.sort_values(
            "learning_rate"
        )

        ax.plot(
            group["learning_rate"],
            group["exact_validation_unscaled_mse"],
            marker="o",
            label=dataset_name.replace("_", " "),
        )

    ax.set_xscale("log")

    ax.set_xlabel(
        "Learning rate"
    )

    ax.set_ylabel(
        "Validation unscaled MSE"
    )

    ax.set_title(
        "Fast Learning-Rate Screening"
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_convergence(
    histories_df,
    output_dir,
):
    for dataset_name, dataset_history in histories_df.groupby(
        "dataset"
    ):
        fig, ax = plt.subplots(
            figsize=(10, 6)
        )

        for learning_rate, group in dataset_history.groupby(
            "learning_rate"
        ):
            group = group.sort_values(
                "epoch"
            )

            ax.plot(
                group["epoch"],
                group["validation_scaled_loss"],
                label=f"{learning_rate:.0e}",
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel(
            "Validation scaled loss"
        )

        ax.set_title(
            f"{dataset_name.replace('_', ' ')} Convergence"
        )

        ax.grid(
            True,
            alpha=0.25,
        )

        ax.legend(
            title="Learning rate"
        )

        fig.tight_layout()

        fig.savefig(
            output_dir
            / f"{dataset_name}_convergence.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================

def main():
    all_results = []
    all_histories = []

    loaded_datasets = {}

    print("=" * 80)
    print("FAST LEARNING-RATE COMPARISON")
    print("=" * 80)

    for dataset_name, config in DATASETS.items():
        print(
            f"\nLoading {dataset_name}..."
        )

        dataset = load_dataset(
            dataset_name,
            config,
        )

        loaded_datasets[
            dataset_name
        ] = dataset

        print(
            f"Training rows used: "
            f"{len(dataset['X_train']):,}"
        )

        print(
            f"Validation rows used: "
            f"{len(dataset['X_val']):,}"
        )

        print(
            f"Features: "
            f"{dataset['X_train'].shape[1]}"
        )

    total_runs = (
        len(loaded_datasets)
        * len(LEARNING_RATES)
    )

    run_number = 0

    for dataset_name, dataset in loaded_datasets.items():
        for learning_rate in LEARNING_RATES:
            run_number += 1

            print("\n" + "-" * 80)

            print(
                f"Run {run_number}/{total_runs}"
            )

            try:
                result, history_df = (
                    run_learning_rate(
                        dataset=dataset,
                        learning_rate=learning_rate,
                    )
                )

                all_results.append(result)
                all_histories.append(history_df)

                print(
                    "Exact validation unscaled MSE: "
                    f"{result['exact_validation_unscaled_mse']:.8g}"
                )

                print(
                    "Best epoch: "
                    f"{result['best_epoch']}"
                )

                print(
                    "Initial gradient norm: "
                    f"{result['initial_gradient_norm']:.8g}"
                )

                print(
                    "Approximate relative update: "
                    f"{result['approximate_relative_update']:.8g}"
                )

            except Exception as exc:
                print(
                    f"Run failed: {exc}"
                )

            # Save partial progress after every run.
            if all_results:
                pd.DataFrame(
                    all_results
                ).to_csv(
                    OUTPUT_DIR
                    / "partial_results.csv",
                    index=False,
                )

    if not all_results:
        raise RuntimeError(
            "No models completed successfully."
        )

    results_df = pd.DataFrame(
        all_results
    )

    results_df["rank_within_dataset"] = (
        results_df
        .groupby("dataset")[
            "exact_validation_unscaled_mse"
        ]
        .rank(
            method="min",
            ascending=True,
        )
    )

    results_df = results_df.sort_values(
        [
            "dataset",
            "rank_within_dataset",
        ]
    )

    results_df.to_csv(
        OUTPUT_DIR
        / "fast_learning_rate_results.csv",
        index=False,
    )

    histories_df = pd.concat(
        all_histories,
        ignore_index=True,
    )

    histories_df.to_csv(
        OUTPUT_DIR
        / "fast_learning_rate_histories.csv",
        index=False,
    )

    plot_validation_performance(
        results_df=results_df,
        output_path=(
            OUTPUT_DIR
            / "learning_rate_comparison.png"
        ),
    )

    plot_convergence(
        histories_df=histories_df,
        output_dir=OUTPUT_DIR,
    )

    print("\n" + "=" * 80)
    print("FINAL SCREENING RESULTS")
    print("=" * 80)

    columns_to_print = [
        "dataset",
        "learning_rate",
        "is_original_learning_rate",
        "exact_validation_unscaled_mse",
        "best_epoch",
        "epochs_completed",
        "initial_gradient_norm",
        "approximate_relative_update",
        "rank_within_dataset",
    ]

    print(
        results_df[
            columns_to_print
        ].to_string(
            index=False
        )
    )

    print("\nBest learning rate per dataset:")

    best_rows = (
        results_df
        .sort_values(
            "exact_validation_unscaled_mse"
        )
        .groupby(
            "dataset",
            as_index=False,
        )
        .first()
    )

    print(
        best_rows[
            [
                "dataset",
                "learning_rate",
                "exact_validation_unscaled_mse",
                "best_epoch",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        f"\nResults saved t<LOCAL_PATH>"
    )


if __name__ == "__main__":
    main()

In [ ]:
import math
import pandas as pd
import joblib
from pathlib import Path

DATASETS = {
    "White_Tailed_Deer": {
        "base": Path("<LOCAL_PATH>"),
        "train_file": "train_processed_100.csv",
        "feature_file": "feature_columns_100.pkl",
    },
    "Mule_Deer": {
        "base": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment"
        ),
        "train_file": "train_processed_60_40.csv",
        "feature_file": "feature_columns_60_40.pkl",
    },
}

BATCH_SIZE = 128
MAX_EPOCHS = 100

rows = []

for dataset_name, config in DATASETS.items():
    train_path = config["base"] / config["train_file"]
    feature_path = config["base"] / config["feature_file"]

    feature_columns = joblib.load(feature_path)

    # Only load one feature column because we just need the row count.
    df = pd.read_csv(
        train_path,
        usecols=[feature_columns[0]],
    )

    n_rows = len(df)
    updates_per_epoch = math.ceil(n_rows / BATCH_SIZE)
    maximum_updates = updates_per_epoch * MAX_EPOCHS

    rows.append({
        "dataset": dataset_name,
        "training_rows": n_rows,
        "batch_size": BATCH_SIZE,
        "updates_per_epoch": updates_per_epoch,
        "maximum_epochs": MAX_EPOCHS,
        "maximum_optimizer_updates": maximum_updates,
    })

summary = pd.DataFrame(rows)

reference_updates = summary[
    "maximum_optimizer_updates"
].min()

summary["updates_relative_to_smaller_dataset"] = (
    summary["maximum_optimizer_updates"]
    / reference_updates
)

print(summary.to_string(index=False))

In [ ]:
import math

TARGET_UPDATES = 30_000
BATCH_SIZE = 128

wtd_rows = 106_989
mule_rows = 38_225

wtd_steps_per_epoch = math.ceil(wtd_rows / BATCH_SIZE)
mule_steps_per_epoch = math.ceil(mule_rows / BATCH_SIZE)

wtd_epochs = math.ceil(TARGET_UPDATES / wtd_steps_per_epoch)
mule_epochs = math.ceil(TARGET_UPDATES / mule_steps_per_epoch)

print("White-tailed deer epochs:", wtd_epochs)
print("Mule deer epochs:", mule_epochs)

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

DATASETS = {
    "White_Tailed_Deer": {
        "base": Path("<LOCAL_PATH>"),
        "train": "train_processed_100.csv",
        "features": "feature_columns_100.pkl",
    },
    "Mule_Deer": {
        "base": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment"
        ),
        "train": "train_processed_60_40.csv",
        "features": "feature_columns_60_40.pkl",
    },
}

rows = []

for dataset_name, config in DATASETS.items():
    feature_columns = list(
        joblib.load(config["base"] / config["features"])
    )

    df = pd.read_csv(
        config["base"] / config["train"],
        usecols=feature_columns,
    )

    X = df.to_numpy(dtype=np.float64)

    feature_std = np.std(X, axis=0)
    row_norm = np.linalg.norm(X, axis=1)

    input_dim = len(feature_columns)
    loss_scale = 1.0 + 0.5 * input_dim

    rows.append({
        "dataset": dataset_name,
        "n_features": input_dim,
        "loss_scale": loss_scale,
        "mean_absolute_value": np.mean(np.abs(X)),
        "rms_feature_value": np.sqrt(np.mean(X ** 2)),
        "median_feature_std": np.median(feature_std),
        "maximum_feature_std": np.max(feature_std),
        "median_row_norm": np.median(row_norm),
        "p95_row_norm": np.quantile(row_norm, 0.95),
        "p99_row_norm": np.quantile(row_norm, 0.99),
        "maximum_row_norm": np.max(row_norm),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

In [ ]:
# =============================================================================
# COMPARE GRADIENT VARIABILITY ACROSS MANY RANDOM BATCHES
# White-Tailed Deer vs. Mule Deer
# =============================================================================

import os
import gc
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Model, layers, regularizers


# =============================================================================
# CONFIGURATION
# =============================================================================

SEED = 13
BATCH_SIZE = 128
N_BATCHES = 200

LAMBDA_DEV = 0.5
L2_REGULARIZATION = 1e-4

OUTPUT_PATH = Path(
    "<LOCAL_PATH>"
)

DATASETS = {
    "White_Tailed_Deer": {
        "base": Path("<LOCAL_PATH>"),
        "train": "train_processed_100.csv",
        "features": "feature_columns_100.pkl",
    },
    "Mule_Deer": {
        "base": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment"
        ),
        "train": "train_processed_60_40.csv",
        "features": "feature_columns_60_40.pkl",
    },
}


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# =============================================================================
# MODEL
# =============================================================================

def build_autoencoder(input_dim, seed):
    tf.keras.utils.set_random_seed(seed)

    reg = regularizers.l2(L2_REGULARIZATION)

    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=reg,
    )(inputs)

    x = layers.Dense(
        16,
        activation="relu",
        kernel_regularizer=reg,
    )(x)

    x = layers.Dense(
        4,
        activation="relu",
        kernel_regularizer=reg,
        name="bottleneck",
    )(x)

    x = layers.Dense(
        16,
        activation="relu",
        kernel_regularizer=reg,
    )(x)

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=reg,
    )(x)

    outputs = layers.Dense(
        input_dim,
        activation="linear",
    )(x)

    return Model(inputs, outputs)


def make_loss(input_dim):
    loss_scale = 1.0 + LAMBDA_DEV * input_dim

    def scaled_mse(y_true, y_pred):
        row_mse = tf.reduce_mean(
            tf.square(y_true - y_pred),
            axis=-1,
        )

        return loss_scale * row_mse

    return scaled_mse, float(loss_scale)


# =============================================================================
# LOAD DATA
# =============================================================================

def load_training_matrix(config):
    feature_columns = list(
        joblib.load(
            config["base"] / config["features"]
        )
    )

    df = pd.read_csv(
        config["base"] / config["train"],
        usecols=feature_columns,
    )

    X = df.to_numpy(dtype=np.float32)

    if not np.isfinite(X).all():
        raise ValueError(
            "Training matrix contains NaN or infinite values."
        )

    return X, feature_columns


# =============================================================================
# GRADIENT ANALYSIS
# =============================================================================

def measure_gradient_batches(
    dataset_name,
    X,
    input_dim,
):
    """
    Measure gradient magnitude across many randomly sampled batches.

    The model weights remain fixed during this diagnostic. Therefore, this
    isolates differences caused by batch composition rather than changes
    caused by training.
    """
    tf.keras.backend.clear_session()
    gc.collect()

    model = build_autoencoder(
        input_dim=input_dim,
        seed=SEED,
    )

    loss_fn, loss_scale = make_loss(input_dim)

    rng = np.random.default_rng(SEED)

    rows = []

    for batch_number in range(1, N_BATCHES + 1):
        replace = len(X) < BATCH_SIZE

        indices = rng.choice(
            len(X),
            size=BATCH_SIZE,
            replace=replace,
        )

        batch = tf.convert_to_tensor(
            X[indices],
            dtype=tf.float32,
        )

        with tf.GradientTape() as tape:
            reconstruction = model(
                batch,
                training=True,
            )

            reconstruction_loss = tf.reduce_mean(
                loss_fn(
                    batch,
                    reconstruction,
                )
            )

            if model.losses:
                regularization_loss = tf.add_n(
                    model.losses
                )
            else:
                regularization_loss = tf.constant(
                    0.0,
                    dtype=tf.float32,
                )

            total_loss = (
                reconstruction_loss
                + regularization_loss
            )

        gradients = tape.gradient(
            total_loss,
            model.trainable_variables,
        )

        gradients = [
            gradient
            for gradient in gradients
            if gradient is not None
        ]

        gradient_norm = float(
            tf.linalg.global_norm(
                gradients
            ).numpy()
        )

        gradient_max_abs = max(
            float(
                tf.reduce_max(
                    tf.abs(gradient)
                ).numpy()
            )
            for gradient in gradients
        )

        gradient_mean_abs = float(
            np.mean([
                float(
                    tf.reduce_mean(
                        tf.abs(gradient)
                    ).numpy()
                )
                for gradient in gradients
            ])
        )

        batch_array = batch.numpy()

        rows.append({
            "dataset": dataset_name,
            "batch_number": batch_number,
            "loss_scale": loss_scale,
            "batch_reconstruction_loss":
                float(reconstruction_loss.numpy()),
            "batch_regularization_loss":
                float(regularization_loss.numpy()),
            "batch_total_loss":
                float(total_loss.numpy()),
            "gradient_global_norm":
                gradient_norm,
            "gradient_max_absolute":
                gradient_max_abs,
            "gradient_mean_absolute":
                gradient_mean_abs,
            "batch_mean_absolute_feature_value":
                float(np.mean(np.abs(batch_array))),
            "batch_max_absolute_feature_value":
                float(np.max(np.abs(batch_array))),
            "batch_mean_row_norm":
                float(
                    np.mean(
                        np.linalg.norm(
                            batch_array,
                            axis=1,
                        )
                    )
                ),
        })

    return pd.DataFrame(rows)


# =============================================================================
# MAIN
# =============================================================================

all_batch_results = []

for dataset_name, config in DATASETS.items():
    print(f"\nAnalyzing {dataset_name}...")

    X, feature_columns = load_training_matrix(
        config
    )

    batch_results = measure_gradient_batches(
        dataset_name=dataset_name,
        X=X,
        input_dim=len(feature_columns),
    )

    all_batch_results.append(batch_results)

gradient_results = pd.concat(
    all_batch_results,
    ignore_index=True,
)

gradient_results.to_csv(
    OUTPUT_PATH,
    index=False,
)

summary = (
    gradient_results
    .groupby("dataset", as_index=False)
    .agg(
        mean_gradient_norm=(
            "gradient_global_norm",
            "mean",
        ),
        sd_gradient_norm=(
            "gradient_global_norm",
            "std",
        ),
        median_gradient_norm=(
            "gradient_global_norm",
            "median",
        ),
        p90_gradient_norm=(
            "gradient_global_norm",
            lambda values: np.quantile(values, 0.90),
        ),
        p95_gradient_norm=(
            "gradient_global_norm",
            lambda values: np.quantile(values, 0.95),
        ),
        p99_gradient_norm=(
            "gradient_global_norm",
            lambda values: np.quantile(values, 0.99),
        ),
        maximum_gradient_norm=(
            "gradient_global_norm",
            "max",
        ),
        mean_batch_loss=(
            "batch_total_loss",
            "mean",
        ),
        sd_batch_loss=(
            "batch_total_loss",
            "std",
        ),
        p95_batch_loss=(
            "batch_total_loss",
            lambda values: np.quantile(values, 0.95),
        ),
        maximum_batch_loss=(
            "batch_total_loss",
            "max",
        ),
    )
)

summary["gradient_coefficient_of_variation"] = (
    summary["sd_gradient_norm"]
    / summary["mean_gradient_norm"]
)

summary["loss_coefficient_of_variation"] = (
    summary["sd_batch_loss"]
    / summary["mean_batch_loss"]
)

print("\n" + "=" * 100)
print("GRADIENT VARIABILITY SUMMARY")
print("=" * 100)
print(summary.to_string(index=False))

print(f"\nDetailed results saved t<LOCAL_PATH>")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATASETS = {
    "White_Tailed_Deer": {
        "history": Path(
            "<LOCAL_PATH>"
        ),
        "learning_rate": 1e-6,
        "training_rows": 106_989,
    },
    "Mule_Deer": {
        "history": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment/"
            "training_history.npy"
        ),
        "learning_rate": 1e-4,
        "training_rows": 38_225,
    },
}

BATCH_SIZE = 128
PATIENCE = 10

rows = []
history_tables = []

for dataset_name, config in DATASETS.items():
    history_path = config["history"]

    if not history_path.exists():
        raise FileNotFoundError(
            f"History file not found for {dataset_name}:\n"
            f"{history_path}"
        )

    history = np.load(
        history_path,
        allow_pickle=True,
    ).item()

    train_loss = np.asarray(
        history["loss"],
        dtype=np.float64,
    )

    val_loss = np.asarray(
        history["val_loss"],
        dtype=np.float64,
    )

    epochs_completed = len(train_loss)
    best_epoch_index = int(np.argmin(val_loss))
    best_epoch = best_epoch_index + 1

    updates_per_epoch = int(
        np.ceil(
            config["training_rows"] / BATCH_SIZE
        )
    )

    updates_at_best_epoch = (
        best_epoch * updates_per_epoch
    )

    total_updates = (
        epochs_completed * updates_per_epoch
    )

    # Examine whether validation loss was still improving near the end.
    final_window_size = min(10, epochs_completed)

    first_final_window_loss = float(
        val_loss[-final_window_size]
    )

    final_loss = float(
        val_loss[-1]
    )

    final_window_change = (
        final_loss - first_final_window_loss
    )

    final_window_percent_change = (
        100.0
        * final_window_change
        / max(abs(first_final_window_loss), 1e-12)
    )

    # Compare the best loss with the final recorded loss.
    best_val_loss = float(
        val_loss[best_epoch_index]
    )

    final_vs_best_percent = (
        100.0
        * (final_loss - best_val_loss)
        / max(abs(best_val_loss), 1e-12)
    )

    # Average absolute epoch-to-epoch movement in validation loss.
    if epochs_completed > 1:
        mean_absolute_val_change = float(
            np.mean(
                np.abs(
                    np.diff(val_loss)
                )
            )
        )

        maximum_absolute_val_change = float(
            np.max(
                np.abs(
                    np.diff(val_loss)
                )
            )
        )
    else:
        mean_absolute_val_change = np.nan
        maximum_absolute_val_change = np.nan

    rows.append({
        "dataset": dataset_name,
        "learning_rate": config["learning_rate"],
        "training_rows": config["training_rows"],
        "batch_size": BATCH_SIZE,
        "updates_per_epoch": updates_per_epoch,
        "epochs_completed": epochs_completed,
        "best_epoch": best_epoch,
        "updates_at_best_epoch": updates_at_best_epoch,
        "total_optimizer_updates": total_updates,
        "initial_train_loss": float(train_loss[0]),
        "final_train_loss": float(train_loss[-1]),
        "initial_validation_loss": float(val_loss[0]),
        "best_validation_loss": best_val_loss,
        "final_validation_loss": final_loss,
        "final_vs_best_percent": final_vs_best_percent,
        "final_10_epoch_percent_change":
            final_window_percent_change,
        "mean_absolute_validation_loss_change":
            mean_absolute_val_change,
        "maximum_absolute_validation_loss_change":
            maximum_absolute_val_change,
        "best_epoch_was_final_epoch":
            best_epoch == epochs_completed,
        "stopped_before_100_epochs":
            epochs_completed < 100,
    })

    history_df = pd.DataFrame({
        "dataset": dataset_name,
        "learning_rate": config["learning_rate"],
        "epoch": np.arange(
            1,
            epochs_completed + 1,
        ),
        "training_loss": train_loss,
        "validation_loss": val_loss,
        "optimizer_updates": (
            np.arange(
                1,
                epochs_completed + 1,
            )
            * updates_per_epoch
        ),
    })

    history_tables.append(history_df)

summary = pd.DataFrame(rows)
all_histories = pd.concat(
    history_tables,
    ignore_index=True,
)

print("\n" + "=" * 100)
print("ORIGINAL TRAINING-HISTORY SUMMARY")
print("=" * 100)
print(summary.to_string(index=False))

all_histories.to_csv(
    "<LOCAL_PATH>"
    "original_training_history_comparison.csv",
    index=False,
)

print(
    "\nDetailed history saved t<LOCAL_PATH>"
    "<LOCAL_PATH>"
    "original_training_history_comparison.csv"
)

In [ ]:
# =============================================================================
# COMPARE ANOMALY STRUCTURE:
# WHITE-TAILED DEER VS. MULE DEER AUTOENCODERS
#
# This script compares:
#   1. Total rows and temporal duration per animal
#   2. Number and fraction of anomalous rows
#   3. Empirical fixes per day
#   4. Anomalous fixes per day
#   5. Number and percentage of anomalous days
#   6. Longest consecutive anomalous-day run
#   7. First-to-last anomaly span
#   8. Reconstruction-error severity above theta
#   9. Whether COUNT, FRAC, or both branches caused classification
#  10. Validation-negative versus test anomaly distributions
#
# IMPORTANT:
#   - This does not tune a new threshold.
#   - It diagnoses why 75, 275, and 550 have different meanings.
#   - Test labels are used only for descriptive summaries.
# =============================================================================

from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = Path(
    "<LOCAL_PATH>"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


DATASETS = {
    "White_Tailed_Deer": {
        "base": Path("<LOCAL_PATH>"),

        "validation_rows": Path(
            "<LOCAL_PATH>"
            "val_processed_100_rows_with_flags_FROZEN.csv"
        ),

        "test_rows": Path(
            "<LOCAL_PATH>"
            "test_processed_100_rows_with_flags_FROZEN.csv"
        ),

        "threshold_file": Path(
            "<LOCAL_PATH>"
            "reconstruction_threshold.pkl"
        ),

        "id_rule_file": Path(
            "<LOCAL_PATH>"
            "id_rule_params.pkl"
        ),

        "positive_test_ids": {
            "A41", "A43", "A52", "A56", "A60",
            "F106", "F107", "F26", "F27", "F30",
            "F48", "F62", "F66", "F73", "F78",
            "F79", "F83", "F87", "F94"
        },

        "negative_test_ids": set(),

        # Expected nominal fix rate, used only for context.
        "nominal_fixes_per_day": 8.0,
    },

    "Mule_Deer": {
        "base": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment"
        ),

        "validation_rows": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment/"
            "val_processed_60_40_rows_with_flags_FROZEN.csv"
        ),

        "test_rows": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment/"
            "test_processed_60_40_rows_with_flags_FROZEN.csv"
        ),

        "threshold_file": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment/"
            "reconstruction_threshold.pkl"
        ),

        "id_rule_file": Path(
            "<LOCAL_PATH>"
            "NONAME_Extension_4.0/60_40_Experiment/"
            "id_rule_params.pkl"
        ),

        "positive_test_ids": {
            1, 2, 6, 8, 11, 12, 15, 16, 17, 18,
            19, 20, 21, 23, 24, 25, 30, 31, 32,
            35, 36, 37, 40, 47, 48, 53, 57, 58,
            61, 62, 64, 65, 67, 68, 69, 71, 72,
            74, 79, 83, 90, 91, 92, 93, 94, 96,
            102, 105, 113, 114, 121, 122, 125,
            130, 141
        },

        "negative_test_ids": {
            5, 9, 14, 22, 26, 29, 44,
            55, 66, 95, 108, 118, 129
        },

        "nominal_fixes_per_day": 4.0,
    },
}


# =============================================================================
# HELPERS
# =============================================================================

def standardize_id(series):
    """
    Convert IDs to strings so comparisons are consistent across files.

    Numeric IDs such as 1.0 are converted to "1".
    """
    values = series.astype(str).str.strip()

    values = values.str.replace(
        r"\.0$",
        "",
        regex=True,
    )

    return values


def standardize_id_set(values):
    return {
        str(value).strip().removesuffix(".0")
        for value in values
    }


def find_datetime_column(df):
    """
    Find and parse the datetime field.
    """
    candidates = [
        "datetime",
        "date",
        "timestamp",
        "utc",
        "time",
    ]

    for column in candidates:
        if column in df.columns:
            parsed = pd.to_datetime(
                df[column],
                errors="coerce",
            )

            if parsed.notna().any():
                return column, parsed

    return None, pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns]",
    )


def longest_consecutive_day_run(dates):
    """
    Calculate the longest sequence of consecutive calendar days.

    Multiple anomalous fixes on the same day count as one anomalous day.
    """
    dates = pd.Series(dates).dropna()

    if dates.empty:
        return 0

    unique_days = np.sort(
        dates.dt.normalize().unique()
    )

    if len(unique_days) == 0:
        return 0

    longest = 1
    current = 1

    for previous, current_day in zip(
        unique_days[:-1],
        unique_days[1:],
    ):
        difference = (
            pd.Timestamp(current_day)
            - pd.Timestamp(previous)
        ).days

        if difference == 1:
            current += 1
            longest = max(longest, current)
        else:
            current = 1

    return int(longest)


def longest_consecutive_flag_run(flags):
    """
    Longest run of consecutive flagged rows.

    This is not necessarily uninterrupted calendar time if fixes are missing,
    so it should be interpreted alongside consecutive anomalous days.
    """
    flags = np.asarray(flags, dtype=bool)

    longest = 0
    current = 0

    for flag in flags:
        if flag:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return int(longest)


def safe_divide(numerator, denominator):
    if denominator is None or not np.isfinite(denominator):
        return np.nan

    if denominator == 0:
        return np.nan

    return float(numerator / denominator)


# =============================================================================
# LOAD ROW-LEVEL OUTPUT
# =============================================================================

def load_row_output(path, dataset_name, split_name):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing row-level output for {dataset_name}, {split_name}:\n"
            f"{path}\n\n"
            "Run the validation/test script first so the "
            "*_rows_with_flags_FROZEN.csv file exists."
        )

    df = pd.read_csv(path)

    required = {
        "id",
        "reconstruction_error",
        "row_flag",
    }

    missing = required - set(df.columns)

    if missing:
        raise KeyError(
            f"{path} is missing columns: {sorted(missing)}"
        )

    df["id"] = standardize_id(df["id"])

    datetime_column, parsed_datetime = find_datetime_column(df)

    df["datetime_standardized"] = parsed_datetime
    df["date_standardized"] = (
        df["datetime_standardized"].dt.normalize()
    )

    if df["row_flag"].dtype != bool:
        df["row_flag"] = (
            df["row_flag"]
            .astype(str)
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            })
        )

    if df["row_flag"].isna().any():
        raise ValueError(
            f"{path} contains invalid row_flag values."
        )

    df["reconstruction_error"] = pd.to_numeric(
        df["reconstruction_error"],
        errors="coerce",
    )

    if df["reconstruction_error"].isna().any():
        raise ValueError(
            f"{path} contains invalid reconstruction errors."
        )

    df["dataset"] = dataset_name
    df["split"] = split_name
    df["datetime_source_column"] = datetime_column

    return df


# =============================================================================
# ID-LEVEL ANOMALY SUMMARY
# =============================================================================

def summarize_ids(
    rows,
    dataset_name,
    split_name,
    theta,
    count_threshold,
    fraction_threshold,
    aggregation_rule,
    nominal_fixes_per_day,
    positive_ids,
    negative_ids,
):
    positive_ids = standardize_id_set(positive_ids)
    negative_ids = standardize_id_set(negative_ids)

    summaries = []

    for animal_id, group in rows.groupby(
        "id",
        sort=False,
    ):
        group = group.sort_values(
            "datetime_standardized",
            na_position="last",
        ).copy()

        n_rows = len(group)
        n_flags = int(group["row_flag"].sum())
        flag_fraction = safe_divide(n_flags, n_rows)

        valid_dates = (
            group["datetime_standardized"]
            .dropna()
        )

        if valid_dates.empty:
            start_datetime = pd.NaT
            end_datetime = pd.NaT
            observed_span_days = np.nan
            unique_observed_days = np.nan
            empirical_fixes_per_day = np.nan
        else:
            start_datetime = valid_dates.min()
            end_datetime = valid_dates.max()

            # Inclusive observed span.
            observed_span_days = (
                end_datetime.normalize()
                - start_datetime.normalize()
            ).days + 1

            unique_observed_days = int(
                valid_dates.dt.normalize().nunique()
            )

            empirical_fixes_per_day = safe_divide(
                n_rows,
                unique_observed_days,
            )

        flagged = group.loc[
            group["row_flag"]
        ].copy()

        flagged_dates = (
            flagged["datetime_standardized"]
            .dropna()
        )

        if flagged_dates.empty:
            first_anomaly = pd.NaT
            last_anomaly = pd.NaT
            anomaly_span_days = 0
            n_anomaly_days = 0
            anomaly_day_fraction = 0.0
            anomaly_fixes_per_observed_day = 0.0
            anomaly_fixes_per_anomaly_day = 0.0
            longest_anomalous_day_run = 0
        else:
            first_anomaly = flagged_dates.min()
            last_anomaly = flagged_dates.max()

            anomaly_span_days = (
                last_anomaly.normalize()
                - first_anomaly.normalize()
            ).days + 1

            n_anomaly_days = int(
                flagged_dates.dt.normalize().nunique()
            )

            anomaly_day_fraction = safe_divide(
                n_anomaly_days,
                unique_observed_days,
            )

            anomaly_fixes_per_observed_day = safe_divide(
                n_flags,
                unique_observed_days,
            )

            anomaly_fixes_per_anomaly_day = safe_divide(
                n_flags,
                n_anomaly_days,
            )

            longest_anomalous_day_run = (
                longest_consecutive_day_run(
                    flagged_dates
                )
            )

        longest_flagged_row_run = (
            longest_consecutive_flag_run(
                group["row_flag"].values
            )
        )

        errors = group[
            "reconstruction_error"
        ].to_numpy(dtype=np.float64)

        flagged_errors = flagged[
            "reconstruction_error"
        ].to_numpy(dtype=np.float64)

        mean_error = float(np.mean(errors))
        median_error = float(np.median(errors))
        p90_error = float(np.quantile(errors, 0.90))
        p95_error = float(np.quantile(errors, 0.95))
        maximum_error = float(np.max(errors))

        if len(flagged_errors):
            mean_flagged_error = float(
                np.mean(flagged_errors)
            )

            median_flagged_error = float(
                np.median(flagged_errors)
            )

            mean_excess_over_theta = float(
                np.mean(flagged_errors - theta)
            )

            median_error_ratio_to_theta = float(
                np.median(flagged_errors / theta)
            )
        else:
            mean_flagged_error = np.nan
            median_flagged_error = np.nan
            mean_excess_over_theta = np.nan
            median_error_ratio_to_theta = np.nan

        count_branch = (
            n_flags >= count_threshold
        )

        fraction_branch = (
            flag_fraction >= fraction_threshold
        )

        if aggregation_rule == "AND":
            predicted_positive = (
                count_branch and fraction_branch
            )
        elif aggregation_rule == "FRAC_ONLY":
            predicted_positive = fraction_branch
        elif aggregation_rule == "COUNT_ONLY":
            predicted_positive = count_branch
        else:
            predicted_positive = (
                count_branch or fraction_branch
            )

        if count_branch and fraction_branch:
            decision_branch = "BOTH"
        elif count_branch:
            decision_branch = "COUNT_ONLY"
        elif fraction_branch:
            decision_branch = "FRAC_ONLY"
        else:
            decision_branch = "NEITHER"

        if animal_id in positive_ids:
            actual_class = "POSITIVE"
        elif animal_id in negative_ids:
            actual_class = "NEGATIVE"
        else:
            actual_class = "UNKNOWN"

        # Empirical days represented by the count threshold.
        if (
            empirical_fixes_per_day is not None
            and np.isfinite(empirical_fixes_per_day)
            and empirical_fixes_per_day > 0
        ):
            threshold_equivalent_days_empirical = (
                count_threshold
                / empirical_fixes_per_day
            )
        else:
            threshold_equivalent_days_empirical = np.nan

        threshold_equivalent_days_nominal = (
            count_threshold
            / nominal_fixes_per_day
        )

        summaries.append({
            "dataset": dataset_name,
            "split": split_name,
            "id": animal_id,
            "actual_class": actual_class,

            "n_rows": int(n_rows),
            "n_flags": int(n_flags),
            "flag_fraction": flag_fraction,

            "start_datetime": start_datetime,
            "end_datetime": end_datetime,
            "observed_span_days": observed_span_days,
            "unique_observed_days": unique_observed_days,
            "empirical_fixes_per_day":
                empirical_fixes_per_day,

            "first_anomaly": first_anomaly,
            "last_anomaly": last_anomaly,
            "anomaly_span_days": anomaly_span_days,
            "n_anomaly_days": n_anomaly_days,
            "anomaly_day_fraction":
                anomaly_day_fraction,
            "anomaly_fixes_per_observed_day":
                anomaly_fixes_per_observed_day,
            "anomaly_fixes_per_anomaly_day":
                anomaly_fixes_per_anomaly_day,

            "longest_anomalous_day_run":
                longest_anomalous_day_run,
            "longest_flagged_row_run":
                longest_flagged_row_run,

            "mean_reconstruction_error": mean_error,
            "median_reconstruction_error": median_error,
            "p90_reconstruction_error": p90_error,
            "p95_reconstruction_error": p95_error,
            "maximum_reconstruction_error": maximum_error,

            "mean_flagged_error": mean_flagged_error,
            "median_flagged_error":
                median_flagged_error,
            "mean_excess_over_theta":
                mean_excess_over_theta,
            "median_flagged_error_ratio_to_theta":
                median_error_ratio_to_theta,

            "theta": float(theta),
            "count_threshold":
                int(count_threshold),
            "fraction_threshold":
                float(fraction_threshold),
            "aggregation_rule":
                aggregation_rule,

            "count_branch_met":
                bool(count_branch),
            "fraction_branch_met":
                bool(fraction_branch),
            "decision_branch":
                decision_branch,
            "predicted_positive":
                bool(predicted_positive),

            "threshold_equivalent_days_nominal":
                threshold_equivalent_days_nominal,
            "threshold_equivalent_days_empirical":
                threshold_equivalent_days_empirical,
        })

    return pd.DataFrame(summaries)


# =============================================================================
# DATASET-LEVEL SUMMARY
# =============================================================================

def summarize_dataset_level(id_summary):
    rows = []

    grouping_columns = [
        "dataset",
        "split",
        "actual_class",
    ]

    for keys, group in id_summary.groupby(
        grouping_columns,
        dropna=False,
    ):
        dataset, split, actual_class = keys

        rows.append({
            "dataset": dataset,
            "split": split,
            "actual_class": actual_class,
            "n_ids": len(group),

            "median_rows_per_id":
                group["n_rows"].median(),
            "median_observed_days":
                group["unique_observed_days"].median(),
            "median_empirical_fixes_per_day":
                group["empirical_fixes_per_day"].median(),

            "median_anomaly_count":
                group["n_flags"].median(),
            "mean_anomaly_count":
                group["n_flags"].mean(),

            "median_flag_fraction":
                group["flag_fraction"].median(),
            "mean_flag_fraction":
                group["flag_fraction"].mean(),

            "median_anomaly_days":
                group["n_anomaly_days"].median(),
            "median_anomaly_day_fraction":
                group["anomaly_day_fraction"].median(),

            "median_anomalies_per_observed_day":
                group[
                    "anomaly_fixes_per_observed_day"
                ].median(),

            "median_longest_anomalous_day_run":
                group[
                    "longest_anomalous_day_run"
                ].median(),

            "median_anomaly_span_days":
                group["anomaly_span_days"].median(),

            "median_error_ratio_to_theta":
                group[
                    "median_flagged_error_ratio_to_theta"
                ].median(),

            "percent_count_branch_met":
                100.0 * group["count_branch_met"].mean(),

            "percent_fraction_branch_met":
                100.0 * group["fraction_branch_met"].mean(),

            "percent_predicted_positive":
                100.0 * group["predicted_positive"].mean(),
        })

    return pd.DataFrame(rows)


# =============================================================================
# COUNT-THRESHOLD SENSITIVITY
# =============================================================================

def count_threshold_sensitivity(
    id_summary,
    candidate_counts,
):
    """
    Show how many IDs cross each count threshold.

    Fraction threshold is not included here because this section is intended
    to isolate the behavior of the count branch.
    """
    rows = []

    for (
        dataset,
        split,
        actual_class
    ), group in id_summary.groupby(
        ["dataset", "split", "actual_class"],
        dropna=False,
    ):
        for count in candidate_counts:
            rows.append({
                "dataset": dataset,
                "split": split,
                "actual_class": actual_class,
                "candidate_count": int(count),
                "n_ids": len(group),
                "n_crossing_count":
                    int((group["n_flags"] >= count).sum()),
                "fraction_crossing_count":
                    float((group["n_flags"] >= count).mean()),
            })

    return pd.DataFrame(rows)


# =============================================================================
# PLOTS
# =============================================================================

def save_anomaly_count_plot(
    test_summary,
    output_path,
):
    fig, ax = plt.subplots(figsize=(11, 7))

    for dataset_name, group in test_summary.groupby(
        "dataset"
    ):
        values = np.sort(
            group["n_flags"].to_numpy()
        )

        ranks = np.arange(
            1,
            len(values) + 1,
        )

        ax.plot(
            ranks,
            values,
            marker="o",
            label=dataset_name.replace("_", " "),
        )

    ax.set_xlabel("Animal rank")
    ax.set_ylabel("Number of anomalous fixes")
    ax.set_title(
        "Distribution of Anomalous Fix Counts by Animal"
    )
    ax.grid(True, alpha=0.25)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_anomaly_day_fraction_plot(
    test_summary,
    output_path,
):
    fig, ax = plt.subplots(figsize=(10, 7))

    labels = []
    data = []

    for dataset_name, group in test_summary.groupby(
        "dataset"
    ):
        labels.append(
            dataset_name.replace("_", " ")
        )

        data.append(
            group["anomaly_day_fraction"]
            .dropna()
            .to_numpy()
        )

    ax.boxplot(
        data,
        tick_labels=labels,
        showmeans=True,
    )

    ax.set_ylabel(
        "Fraction of observed days containing anomalies"
    )
    ax.set_title(
        "Temporal Anomaly Burden by Dataset"
    )
    ax.grid(True, axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_anomaly_persistence_plot(
    test_summary,
    output_path,
):
    fig, ax = plt.subplots(figsize=(10, 7))

    for dataset_name, group in test_summary.groupby(
        "dataset"
    ):
        ax.scatter(
            group["n_anomaly_days"],
            group["longest_anomalous_day_run"],
            alpha=0.8,
            label=dataset_name.replace("_", " "),
        )

    ax.set_xlabel("Number of anomalous days")
    ax.set_ylabel(
        "Longest consecutive anomalous-day run"
    )
    ax.set_title(
        "Anomaly Frequency Versus Temporal Persistence"
    )
    ax.grid(True, alpha=0.25)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_error_severity_plot(
    test_summary,
    output_path,
):
    fig, ax = plt.subplots(figsize=(10, 7))

    labels = []
    data = []

    for dataset_name, group in test_summary.groupby(
        "dataset"
    ):
        values = (
            group["median_flagged_error_ratio_to_theta"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .to_numpy()
        )

        if len(values):
            labels.append(
                dataset_name.replace("_", " ")
            )
            data.append(values)

    ax.boxplot(
        data,
        tick_labels=labels,
        showmeans=True,
    )

    ax.axhline(
        1.0,
        linestyle="--",
        linewidth=1,
    )

    ax.set_ylabel(
        "Median flagged error ÷ row threshold"
    )
    ax.set_title(
        "Severity of Flagged Reconstruction Errors"
    )
    ax.grid(True, axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================

all_id_summaries = []
all_row_files = []

for dataset_name, config in DATASETS.items():
    print("\n" + "=" * 90)
    print(dataset_name)
    print("=" * 90)

    threshold_info = joblib.load(
        config["threshold_file"]
    )

    rule_info = joblib.load(
        config["id_rule_file"]
    )

    theta = float(
        threshold_info["theta"]
    )

    count_threshold = int(
        rule_info["count"]
    )

    fraction_threshold = float(
        rule_info["frac"]
    )

    aggregation_rule = str(
        rule_info.get(
            "aggregation_rule",
            "OR",
        )
    ).upper()

    print(f"Theta: {theta:.6g}")
    print(f"Count threshold: {count_threshold}")
    print(f"Fraction threshold: {fraction_threshold}")
    print(f"Aggregation rule: {aggregation_rule}")

    for split_name, path_key in [
        ("validation_negative", "validation_rows"),
        ("test", "test_rows"),
    ]:
        rows = load_row_output(
            path=config[path_key],
            dataset_name=dataset_name,
            split_name=split_name,
        )

        all_row_files.append(rows)

        id_summary = summarize_ids(
            rows=rows,
            dataset_name=dataset_name,
            split_name=split_name,
            theta=theta,
            count_threshold=count_threshold,
            fraction_threshold=fraction_threshold,
            aggregation_rule=aggregation_rule,
            nominal_fixes_per_day=
                config["nominal_fixes_per_day"],
            positive_ids=
                config["positive_test_ids"],
            negative_ids=
                config["negative_test_ids"],
        )

        all_id_summaries.append(
            id_summary
        )


# Combine ID-level results.
id_summary_all = pd.concat(
    all_id_summaries,
    ignore_index=True,
)

id_summary_all.to_csv(
    OUTPUT_DIR / "id_level_anomaly_comparison.csv",
    index=False,
)


# Dataset-level results.
dataset_summary = summarize_dataset_level(
    id_summary_all
)

dataset_summary.to_csv(
    OUTPUT_DIR / "dataset_level_anomaly_summary.csv",
    index=False,
)


# Candidate count thresholds.
candidate_counts = [
    25,
    50,
    75,
    100,
    150,
    200,
    275,
    350,
    450,
    550,
    650,
    750,
]

count_sensitivity = count_threshold_sensitivity(
    id_summary=id_summary_all,
    candidate_counts=candidate_counts,
)

count_sensitivity.to_csv(
    OUTPUT_DIR / "count_threshold_sensitivity.csv",
    index=False,
)


# Decision-branch summary.
decision_branch_summary = (
    id_summary_all
    .groupby(
        [
            "dataset",
            "split",
            "actual_class",
            "decision_branch",
        ],
        dropna=False,
    )
    .size()
    .rename("n_ids")
    .reset_index()
)

decision_branch_summary.to_csv(
    OUTPUT_DIR / "decision_branch_summary.csv",
    index=False,
)


# Focused test results.
test_summary = id_summary_all[
    id_summary_all["split"] == "test"
].copy()

test_summary.to_csv(
    OUTPUT_DIR / "test_id_anomaly_comparison.csv",
    index=False,
)


# Validation-negative results.
validation_summary = id_summary_all[
    id_summary_all["split"]
    == "validation_negative"
].copy()

validation_summary.to_csv(
    OUTPUT_DIR
    / "validation_negative_anomaly_comparison.csv",
    index=False,
)


# Save plots.
save_anomaly_count_plot(
    test_summary=test_summary,
    output_path=(
        OUTPUT_DIR
        / "anomalous_fix_count_distribution.png"
    ),
)

save_anomaly_day_fraction_plot(
    test_summary=test_summary,
    output_path=(
        OUTPUT_DIR
        / "anomaly_day_fraction_comparison.png"
    ),
)

save_anomaly_persistence_plot(
    test_summary=test_summary,
    output_path=(
        OUTPUT_DIR
        / "anomaly_temporal_persistence.png"
    ),
)

save_error_severity_plot(
    test_summary=test_summary,
    output_path=(
        OUTPUT_DIR
        / "flagged_error_severity.png"
    ),
)


# Console output.
print("\n" + "=" * 110)
print("DATASET-LEVEL ANOMALY SUMMARY")
print("=" * 110)

print(
    dataset_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("DECISION BRANCH SUMMARY")
print("=" * 110)

print(
    decision_branch_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("KEY TEST MEDIANS")
print("=" * 110)

key_columns = [
    "dataset",
    "actual_class",
    "n_rows",
    "n_flags",
    "flag_fraction",
    "unique_observed_days",
    "empirical_fixes_per_day",
    "n_anomaly_days",
    "anomaly_day_fraction",
    "longest_anomalous_day_run",
    "anomaly_span_days",
    "median_flagged_error_ratio_to_theta",
    "decision_branch",
    "predicted_positive",
]

print(
    test_summary[
        key_columns
    ]
    .sort_values(
        [
            "dataset",
            "actual_class",
            "n_flags",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .to_string(
        index=False
    )
)


print(
    f"\nResults saved t<LOCAL_PATH>"
)

In [ ]:
# ── PRINT SELECTED TEST-LEVEL RESULTS ─────────────────────────────────────────
print_columns = [
    "dataset",
    "actual_class",
    "id",
    "n_rows",
    "n_flags",
    "flag_fraction",
    "predicted_positive",
]

print("\n" + "=" * 110)
print("TEST ANOMALY RESULTS BY ID")
print("=" * 110)

print(
    test_summary[
        print_columns
    ]
    .sort_values(
        by=[
            "dataset",
            "actual_class",
            "n_flags",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .to_string(
        index=False,
        formatters={
            "flag_fraction": lambda value: f"{value:.4f}",
        },
    )
)
# ── PRINT SUMMARY BY DATASET AND ACTUAL CLASS ─────────────────────────────────
class_summary = (
    test_summary
    .groupby(
        ["dataset", "actual_class"],
        as_index=False,
    )
    .agg(
        n_ids=("id", "nunique"),
        median_n_rows=("n_rows", "median"),
        mean_n_rows=("n_rows", "mean"),
        median_n_flags=("n_flags", "median"),
        mean_n_flags=("n_flags", "mean"),
        median_flag_fraction=("flag_fraction", "median"),
        mean_flag_fraction=("flag_fraction", "mean"),
        n_predicted_positive=("predicted_positive", "sum"),
        percent_predicted_positive=(
            "predicted_positive",
            lambda values: 100.0 * values.mean(),
        ),
    )
)

print("\n" + "=" * 110)
print("SUMMARY BY DATASET AND ACTUAL CLASS")
print("=" * 110)

print(
    class_summary.to_string(
        index=False,
        formatters={
            "median_n_rows": lambda value: f"{value:.1f}",
            "mean_n_rows": lambda value: f"{value:.1f}",
            "median_n_flags": lambda value: f"{value:.1f}",
            "mean_n_flags": lambda value: f"{value:.1f}",
            "median_flag_fraction": lambda value: f"{value:.4f}",
            "mean_flag_fraction": lambda value: f"{value:.4f}",
            "percent_predicted_positive": lambda value: f"{value:.1f}%",
        },
    )
)



## Ensuring consistency with Reconstruction Errors in this sensitivity analysis comparative to the OG analysis

<LOCAL_PATH>
✅ Wrote: <LOCAL_PATH>
✅ Wrote: <LOCAL_PATH>

In [ ]:
row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta
# Save a processed test file that includes row_flag (so contrib script can use it)
df_test_for_contrib = df.copy()
df_test_for_contrib["row_flag"] = row_flag.astype(bool)
df_test_for_contrib["reconstruction_error"] = row_err.astype(float)

PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS2.csv")
df_test_for_contrib.to_csv(PROC_TEST_WITH_FLAGS, index=False)
print("✅ PROC_TEST_WITH_FLAGS2 =", PROC_TEST_WITH_FLAGS)


In [ ]:
# ===== AE Feature Contributions — holdout-friendly (manuscript-ready, with TP/FN) =====
# UPDATED to plug directly into your AE testing run that already computed:
#   recon, pred_ids, MIN_ROWS_PER_ID, AGGREGATION_RULE, COUNT, FRAC,
#   plus your test id lists: actual_cwd_instances_test / unaffected_cwd_instances_test
#
# Key updates vs your original:
#   1) base/paths point to your Genotype folder (edit if needed)
#   2) PROC_TEST points to *_WITH_FLAGS.csv (written right after row_flag creation)
#   3) TRY_PREDICT_RECON = False (uses recon you already computed)
#   4) pos_set/neg_set + pred_ids injected so TP/FN partitions match frozen-rule outputs
#   5) removed duplicate function definitions near the bottom

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import joblib

# ----------------------------
# Config (point to your AE test run)
# ----------------------------
base   = Path("<LOCAL_PATH>")
outdir = base / "ae_holdout_results"
outdir.mkdir(parents=True, exist_ok=True)

# Processed test file MUST include row_flag (recommended: write *_WITH_FLAGS.csv after AE testing)
# If you followed the earlier step, you created:
#   PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS.csv")
# If you already have that variable in-session, this will use it; otherwise it falls back.
try:
    PROC_TEST = Path(PROC_TEST_WITH_FLAGS)  # created in AE testing step
except NameError:
    PROC_TEST = base / "test_processed_100_WITH_FLAGS.csv"  # rename if yours differs

SCA_PATH  = base / "scaler_100.pkl"             # optional; falls back to scaled units if missing
FEAT_PATH = base / "feature_columns_100.pkl"    # frozen feature order

# Use recon from upstream (AE testing)
TRY_PREDICT_RECON = False
BATCH = 256  # only used if TRY_PREDICT_RECON=True

# Row-level flagging (only used if PROC_TEST has no 'row_flag' column)
ROW_FLAG_TAU  = None
ROW_FLAG_PCTL = 95.0

# ID aggregation params (use frozen values from AE testing if present)
try:
    MIN_ROWS_PER_ID = int(MIN_ROWS_PER_ID)
    AGGREGATION_RULE = str(AGGREGATION_RULE).upper()
    COUNT = int(COUNT)
    FRAC  = float(FRAC)
except NameError:
    MIN_ROWS_PER_ID = 10
    AGGREGATION_RULE = "OR"
    COUNT = 5
    FRAC  = 0.05

# Ground-truth on holdout (prefer explicit lists from your AE testing script)
try:
    pos_set = set(actual_cwd_instances_test)
    neg_set = set(unaffected_cwd_instances_test)
    ASSUME_TEST_ALL_POS = (len(neg_set) == 0)
except NameError:
    ASSUME_TEST_ALL_POS = True
    pos_set, neg_set = None, None

# Prefer pred_ids computed from frozen rule (so partitions are identical)
try:
    pred_ids = sorted(set(pred_ids))
except NameError:
    pred_ids = None

# ----------------------------
# Load processed test & feature list
# ----------------------------
df_test    = pd.read_csv(PROC_TEST)
feat_names = joblib.load(FEAT_PATH)  # exact order used for training

missing_feats = [c for c in feat_names if c not in df_test.columns]
if missing_feats:
    raise KeyError(
        f"Processed test missing {len(missing_feats)} features. Example: {missing_feats[:10]}"
    )

X_test   = df_test[feat_names].to_numpy(dtype=np.float32)
ids_rows = df_test["id"].to_numpy()

has_row_flag = "row_flag" in df_test.columns

# ----------------------------
# Reconstructions & residuals
# ----------------------------
try:
    _ = recon  # use precomputed recon if provided
except NameError:
    if not TRY_PREDICT_RECON:
        raise ValueError("No 'recon' provided and TRY_PREDICT_RECON is False.")
    try:
        _ = autoencoder
    except NameError:
        raise ValueError("To predict 'recon' on the fly, define 'autoencoder' in the environment.")
    recon = autoencoder.predict(X_test, batch_size=BATCH, verbose=0)

recon = np.asarray(recon, float)
if X_test.shape != recon.shape:
    raise ValueError(f"Shape mismatch: X_test {X_test.shape} vs recon {recon.shape}")

residual_sq = (X_test - recon) ** 2     # (N_rows, n_features)
resid_total = residual_sq.sum(axis=1)   # per-row total residual (τ)

# ----------------------------
# Build the 'rows' frame for masks
# ----------------------------
rows = pd.DataFrame({"id": ids_rows})
if has_row_flag:
    rows["row_flag"] = df_test["row_flag"].astype(bool).to_numpy()
else:
    if ROW_FLAG_TAU is not None:
        rows["row_flag"] = resid_total > ROW_FLAG_TAU
    else:
        thr = np.nanpercentile(resid_total, ROW_FLAG_PCTL)
        rows["row_flag"] = resid_total >= thr

mask_all  = np.ones(len(rows), dtype=bool)
mask_flag = rows["row_flag"].to_numpy(bool)
mask_eval = mask_all.copy()  # whole holdout

# ----------------------------
# Per-feature contribution helper (→ %)
# ----------------------------
def contrib_pct(rsq, mask):
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan, dtype=float)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s) * 100.0 if s > 0 else np.full(rsq.shape[1], np.nan, dtype=float)

contrib_all   = contrib_pct(residual_sq, mask_all)
contrib_flag  = contrib_pct(residual_sq, mask_flag)
contrib_eval  = contrib_pct(residual_sq, mask_eval)

# ----------------------------
# Erraticness across IDs (median per-feature SD in %-points)
# ----------------------------
id_to_rows = defaultdict(list)
for r, i in enumerate(rows["id"].to_numpy()):
    id_to_rows[i].append(r)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, dtype=int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full(residual_sq.shape[1], np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])  # (n_ids, n_features)
    std_per_feat_pp = np.nanstd(M, axis=0)
    return dict(median_feature_sd_pp=float(np.nanmedian(std_per_feat_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

# ----------------------------
# Simple concentration proxies
# ----------------------------
def topk_share_pct(arr, k=3):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    a_sorted = np.sort(a)[::-1]
    return float(np.sum(a_sorted[:min(k, a_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ----------------------------
# Per-feature table
# ----------------------------
df_contrib = pd.DataFrame({
    "feature":    feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]

print("\n=== AE Feature Contributions (% of reconstruction error) — Holdout ===")
print(df_contrib.sort_values("Flagged_%", ascending=False).head(15).to_string(
    index=False, float_format=lambda x: f"{x:.2f}"
))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",     "Top3_share_%": top3_all,  "Top5_share_%": top5_all,
     "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows", "Top3_share_%": top3_flag, "Top5_share_%": top5_flag,
     "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",     "Top3_share_%": top3_eval, "Top5_share_%": top5_eval,
     "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl.loc[1, "Top3_share_%"] - summary_tbl.loc[0, "Top3_share_%"]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl.loc[1, "Top5_share_%"] - summary_tbl.loc[0, "Top5_share_%"]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl.loc[1, "Median_feat_SD_pp"] - summary_tbl.loc[0, "Median_feat_SD_pp"]

print("\n=== Simple concentration & variability summary (holdout) ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ----------------------------
# Category roll-up (movement story)
# ----------------------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low", "est", "high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature":   feat_names,
    "category":  [categorize(f) for f in feat_names],
    "All_%":     contrib_all,
    "Flagged_%": contrib_flag,
    "Eval_%":    contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%", "Flagged_%", "Eval_%"]]
              .sum()
              .sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]

print("\n=== Category roll-up (% of recon error) — Holdout ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ----------------------------
# Build TP / FP / FN / TN partitions (ID-level)
# ----------------------------
# 1) Ground-truth ID sets (prefer injected pos_set/neg_set)
if (pos_set is None) or (neg_set is None):
    if (not ASSUME_TEST_ALL_POS) and ("CWD_label" in df_test.columns):
        id_lab = df_test.groupby("id")["CWD_label"].max()
        pos_set = set(id_lab.index[id_lab == 1])
        neg_set = set(id_lab.index[id_lab == 0])
    else:
        pos_set = set(df_test["id"].unique())
        neg_set = set()

# 2) Aggregate row_flags → predicted positive IDs (unless pred_ids already provided)
def aggregate_ids_from_rowflags(row_flags_bool, ids_series,
                                MIN_ROWS_PER_ID=MIN_ROWS_PER_ID,
                                AGGREGATION_RULE=AGGREGATION_RULE,
                                COUNT=COUNT, FRAC=FRAC):
    tmp = pd.DataFrame({"id": np.asarray(ids_series), "row_flag": row_flags_bool.astype(int)})
    g = tmp.groupby("id", sort=False)
    n_rows = g.size()
    n_flags = g["row_flag"].sum()
    flag_fr = n_flags / n_rows

    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
    n_flags_elig = n_flags.reindex(eligible_idx)
    flag_fr_elig = flag_fr.reindex(eligible_idx)

    if AGGREGATION_RULE == "AND":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "FRAC_ONLY":
        pred = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "COUNT_ONLY":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index)
    else:  # OR
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    return sorted(pred), eligible_idx

if pred_ids is None:
    pred_ids, eligible_idx = aggregate_ids_from_rowflags(rows["row_flag"].to_numpy(bool), rows["id"].to_numpy())
else:
    # eligible IDs should follow the same MIN_ROWS rule; recompute eligibility only
    tmp = pd.DataFrame({"id": rows["id"].to_numpy()})
    n_rows = tmp.groupby("id", sort=False).size()
    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index

# 3) Confusion partitions on eligible IDs
eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
TP_ids = [i for i in eval_ids if (i in pos_set) and (i in pred_ids)]
FP_ids = [i for i in eval_ids if (i in neg_set) and (i in pred_ids)]
FN_ids = [i for i in eval_ids if (i in pos_set) and (i not in pred_ids)]
TN_ids = [i for i in eval_ids if (i in neg_set) and (i not in pred_ids)]

rid = rows["id"]
row_flag_arr = rows["row_flag"].to_numpy(bool)

m_tp = rid.isin(TP_ids).to_numpy()
m_fp = rid.isin(FP_ids).to_numpy()
m_fn = rid.isin(FN_ids).to_numpy()
m_tn = rid.isin(TN_ids).to_numpy()

m_tp_flagged = m_tp & row_flag_arr
m_fp_flagged = m_fp & row_flag_arr
m_fn_unflagged = m_fn & (~row_flag_arr)
m_tn_unflagged = m_tn & (~row_flag_arr)

# ----------------------------
# Per-class contribution tables
# ----------------------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False)
              .reset_index(drop=True))

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

all_series = pd.Series(contrib_all, index=feat_names, name="All_%")

def top_with_delta(df_, label, k=10):
    col = f"{label}_%"
    if df_[col].notna().any():
        out = (df_.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[col] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    else:
        print(f"\n=== Top {k} features — {label} ===\n(no rows)")

top_with_delta(tp_tbl, "TP")
top_with_delta(fp_tbl, "FP")
top_with_delta(fn_tbl, "FN")
top_with_delta(tn_tbl, "TN")

# Save per-class CSVs
tp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tp.csv", index=False)
fp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fp.csv", index=False)
fn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fn.csv", index=False)
tn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tn.csv", index=False)

# Save ALL per-class features (merge All_% + delta for every feature)
for label, df_ in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    if df_ is not None and not df_.empty:
        merged = df_.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                           on="feature", how="left")
        merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
        merged.to_csv(outdir / f"ae_all_features_{label}.csv", index=False)
        print(f"✅ Saved full feature list for {label}: {outdir}/ae_all_features_{label}.csv")

# ----------------------------
# Movement–residual linkage (inverse-scaling numeric features only)
# ----------------------------
try:
    scaler = joblib.load(SCA_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")

    # Prefer the scaler's known feature names if present (skips one-hot/cats not seen by scaler)
    scaler_feats = list(getattr(scaler, "feature_names_in_", []))
    num_cols = [c for c in feat_names if c in scaler_feats] if scaler_feats else feat_names
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
    units_label = "raw units"
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test[feat_names].copy()
    units_label = "scaled units"

# Add IDs + row flags
df_raw["id"] = rows["id"].values
df_raw["row_flag"] = rows["row_flag"].values

# Collapsed movement metrics
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows":       int(len(tmp)),
        "mean_speed":   float(tmp["speed"].mean())   if len(tmp) else np.nan,
        "mean_turning": float(tmp["turning"].mean()) if len(tmp) else np.nan,
        "mean_sinu":    float(tmp["sinu"].mean())    if len(tmp) else np.nan
    }, name=label)

summ_pieces = [
    group_summary(mask_all, "All"),
    group_summary(mask_flag, "Flagged"),
]
if m_tp.any(): summ_pieces.append(group_summary(m_tp, "TP"))
if m_fn.any(): summ_pieces.append(group_summary(m_fn, "FN"))
if m_fp.any(): summ_pieces.append(group_summary(m_fp, "FP"))
if m_tn.any(): summ_pieces.append(group_summary(m_tn, "TN"))

summaries = pd.concat(summ_pieces, axis=1).T

# %diff vs TP (if TP exists)
if ("TP" in summaries.index) and (summaries.loc["TP", "n_rows"] > 0):
    for col in ["mean_speed", "mean_turning", "mean_sinu"]:
        base_val = summaries.loc["TP", col]
        for grp in summaries.index:
            if grp == "TP":
                continue
            summaries.loc[grp, f"{col}_vsTP_%diff"] = (
                (summaries.loc[grp, col] - base_val) / base_val * 100
                if (pd.notna(base_val) and base_val != 0) else np.nan
            )

print(f"\n=== Movement summaries by class ({units_label}) — Holdout ===")
print(summaries.round(3))
summaries.to_csv(outdir / f"ae_holdout_movement_summaries_by_class_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                 index=True)

# Direct per-variable summaries
vars_of_interest = [
    "abs_rel_angle", "est", "low", "high",
    "rolling_abs_rel_angle_kurt", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_velo_kurt", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "sinuosity_28pt", "velo"
]
present = [v for v in vars_of_interest if v in df_raw.columns]

def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, present]
    stats = {f"{v}_mean": tmp[v].mean() for v in present}
    stats.update({f"{v}_sd": tmp[v].std() for v in present})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summ_vars_pieces = [
    group_summary_vars(mask_all,  "All"),
    group_summary_vars(mask_flag, "Flagged"),
]
if m_tp.any(): summ_vars_pieces.append(group_summary_vars(m_tp, "TP"))
if m_fn.any(): summ_vars_pieces.append(group_summary_vars(m_fn, "FN"))
if m_fp.any(): summ_vars_pieces.append(group_summary_vars(m_fp, "FP"))
if m_tn.any(): summ_vars_pieces.append(group_summary_vars(m_tn, "TN"))

summaries_vars = pd.concat(summ_vars_pieces, axis=1).T

# diffs vs TP (only if TP exists)
if "TP" in summaries_vars.index and summaries_vars.loc["TP", "n_rows"] > 0:
    for v in present:
        base = summaries_vars.loc["TP", f"{v}_mean"]
        for grp in summaries_vars.index:
            if grp == "TP":
                continue
            summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - base
            summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
                (summaries_vars.loc[grp, f"{v}_mean"] - base) / base * 100
                if (pd.notna(base) and base != 0) else np.nan
            )

print("\n=== Per-variable movement summaries — Holdout ===")
print(summaries_vars.round(3))
summaries_vars.to_csv(outdir / f"ae_holdout_movement_summaries_by_variable_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# Residual correlation with movement metrics
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed":       df_raw["speed"],
    "turning":     df_raw["turning"],
    "sinu":        df_raw["sinu"],
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))
movement_links.to_csv(outdir / f"ae_holdout_movement_residual_correlations_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# ----------------------------
# Save the main contribution tables
# ----------------------------
(df_contrib.sort_values("Flagged_%", ascending=False)
          .to_csv(outdir / "ae_holdout_feature_contributions_by_feature.csv", index=False))
cat_roll.to_csv(outdir / "ae_holdout_feature_contributions_by_category.csv")
summary_tbl.to_csv(outdir / "ae_holdout_feature_contrib_summary_simple.csv", index=False)

print(f"\n💾 Saved tables to: {outdir}")
print(f"IDs — TP:{len(TP_ids)}  FN:{len(FN_ids)}  FP:{len(FP_ids)}  TN:{len(TN_ids)}  | Eligible:{len(set(eval_ids))}")


In [ ]:
# make_ae_feature_contributions_heatmap_BLUE.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    r"<LOCAL_PATH>"
)

CSV_PATH = (
    BASE
    / "ae_holdout_results/"
    / "ae_holdout_feature_contributions_by_feature.csv"
)

OUT_DIR = (
    BASE
    / "ae_holdout_results"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = (
    OUT_DIR
    / "ae_feature_contributions_heatmap_single_file"
)


# =============================================================================
# Model labels
#
# MODEL_LABEL is retained internally.
# MODEL_DISPLAY_LABEL controls only how the column heading appears in the figure.
# =============================================================================

MODEL_LABEL = (
    "White-Tailed Deer Autoencoder Model"
)

MODEL_DISPLAY_LABEL = (
    "Full Holdout"
)


# =============================================================================
# Load feature-contribution data
# =============================================================================

if not CSV_PATH.exists():

    raise FileNotFoundError(
        f"Input file not foun<LOCAL_PATH>"
        f"{CSV_PATH}"
    )


df = pd.read_csv(
    CSV_PATH
)


required_columns = {
    "feature",
    "All_%"
}


missing_columns = required_columns.difference(
    df.columns
)


if missing_columns:

    raise ValueError(
        f"Missing columns: "
        f"{sorted(missing_columns)}\n"
        f"Available columns: "
        f"{list(df.columns)}"
    )


df = df[
    [
        "feature",
        "All_%"
    ]
].copy()


df["All_%"] = pd.to_numeric(
    df["All_%"],
    errors="coerce"
)


df["model"] = MODEL_LABEL


# =============================================================================
# Convert to heatmap format
# =============================================================================

heatmap_df = df.pivot(
    index="feature",
    columns="model",
    values="All_%"
)


# =============================================================================
# Feature ordering and display labels
# =============================================================================

pretty_order = [

    # Velocity metrics
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction and turning-angle metrics
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path-shape metrics
    "sinuosity",
    "sinuosity_28pt",
    "sinuosity_56pt",

    # Space-use metrics
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "sinuosity_28pt":
        "Sinuosity",

    "sinuosity_56pt":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [
    feature
    for feature in pretty_order
    if feature in heatmap_df.index
]


heatmap_df = heatmap_df.reindex(
    available_pretty_order
)


heatmap_df.index = [
    pretty_map.get(
        feature,
        feature
    )
    for feature in heatmap_df.index
]


heatmap_df = heatmap_df.apply(
    pd.to_numeric,
    errors="coerce"
)


# =============================================================================
# Replace model column heading with publication-friendly display label
# =============================================================================

heatmap_df.columns = [
    MODEL_DISPLAY_LABEL
]


# =============================================================================
# Print numerical values for verification
# =============================================================================

print(
    "\nFeature-contribution table:"
)


print(
    heatmap_df
    .round(1)
    .to_string()
)


# =============================================================================
# Determine color scale
# =============================================================================

data_values = heatmap_df.to_numpy(
    dtype=float
)


if not np.isfinite(
    data_values
).any():

    raise ValueError(
        "The heatmap contains no finite "
        "feature-contribution values."
    )


true_min = float(
    np.nanmin(
        data_values
    )
)


true_max = float(
    np.nanmax(
        data_values
    )
)


vmin = 0.0


vmax = float(
    np.ceil(
        true_max
    )
)


if vmax <= vmin:

    vmax = 5.0


print(
    f"\nData range: "
    f"{true_min:.1f}% to "
    f"{true_max:.1f}%"
)


print(
    f"Plotting range: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Build discrete blue color boundaries
#
# 0% to 5% receives a light blue.
# Larger feature contributions become progressively darker.
# =============================================================================

candidate_bounds = [
    vmin,
    5,
    10,
    15,
    20,
    30,
    40,
    50,
    vmax
]


bounds = [
    float(bound)
    for bound in candidate_bounds
    if vmin <= bound <= vmax
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(bounds)
)


if len(bounds) < 2:

    bounds = [
        vmin,
        vmax
    ]


print(
    f"Color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Create one blue shade for each discrete interval
#
# Avoid the nearly white lower portion of the Blues colormap.
# The highest interval receives the darkest blue.
# =============================================================================

base_cmap = mpl.colormaps[
    "Blues"
]


bin_midpoints = np.array(
    [
        (
            lower +
            upper
        ) / 2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


MIN_CMAP_POSITION = 0.15

MAX_CMAP_POSITION = 1.00


if vmax > 0:

    relative_positions = (
        bin_midpoints /
        vmax
    )

else:

    relative_positions = np.zeros_like(
        bin_midpoints
    )


relative_positions = np.clip(
    relative_positions,
    0,
    1
)


color_positions = (
    MIN_CMAP_POSITION
    +
    relative_positions
    *
    (
        MAX_CMAP_POSITION
        -
        MIN_CMAP_POSITION
    )
)


# =============================================================================
# Force first and last intervals to intended endpoints
# =============================================================================

if len(color_positions) > 0:

    color_positions[
        0
    ] = MIN_CMAP_POSITION

    color_positions[
        -1
    ] = MAX_CMAP_POSITION


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_blues"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


print(
    "\nColor intervals:"
)


for index, (
    lower,
    upper
) in enumerate(
    zip(
        bounds[:-1],
        bounds[1:]
    )
):

    print(
        f"  {lower:>6.1f}% to "
        f"{upper:>6.1f}% "
        f"-> colormap position "
        f"{color_positions[index]:.3f}"
    )


# =============================================================================
# Figure dimensions
#
# Match the proportions of the CWD+ vs. CWD- heatmap.
# The overall figure remains wide enough for long feature names while the
# actual colored data column stays compact.
# =============================================================================

n_rows, n_cols = heatmap_df.shape


FIG_WIDTH = 8.0

ROW_HEIGHT = 0.62


FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.0,
    8.5,
    12.0
)


# =============================================================================
# Font sizes
# =============================================================================

AX_LABEL_FS = 26

X_TICK_FS = 19

Y_TICK_FS = 20

ANNOT_FS = 20

CBAR_LABEL_FS = 24

CBAR_TICK_FS = 18


# =============================================================================
# Create figure
#
# Fixed axes are used instead of constrained_layout so that both heatmaps
# have consistent column and colorbar proportions.
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# =============================================================================
# Main heatmap axis
#
# [left, bottom, width, height]
# =============================================================================

ax = fig.add_axes(
    [
        0.52,   # left
        0.12,   # bottom
        0.18,   # width
        0.80    # height
    ]
)


# =============================================================================
# Colorbar axis
# =============================================================================

cax = fig.add_axes(
    [
        0.76,   # left
        0.12,   # bottom
        0.035,  # width
        0.80    # height
    ]
)


# =============================================================================
# Create heatmap
# =============================================================================

hm = sns.heatmap(
    heatmap_df,

    cmap=discrete_cmap,
    norm=norm,

    annot=True,
    fmt=".1f",

    annot_kws={
        "fontsize": ANNOT_FS,
        "fontfamily": "Times New Roman",
        "fontweight": "bold"
    },

    linewidths=1.0,
    linecolor="white",

    ax=ax,

    cbar=True,
    cbar_ax=cax
)


# =============================================================================
# Allow cell to use controlled axis width
# =============================================================================

ax.set_aspect(
    "auto"
)


# =============================================================================
# Determine darkest blue interval
# =============================================================================

darkest_threshold = (
    bounds[-2]
    if len(bounds) >= 2
    else vmax
)


# =============================================================================
# Force ALL annotations to the same Times New Roman bold style
# =============================================================================

flat_values = (
    heatmap_df
    .to_numpy()
    .flatten()
)


for text, value in zip(
    ax.texts,
    flat_values
):

    text.set_fontfamily(
        "Times New Roman"
    )

    text.set_fontsize(
        ANNOT_FS
    )

    text.set_fontweight(
        "bold"
    )


    if not np.isfinite(
        value
    ):

        continue


    # White text only on the darkest blue interval
    if value >= darkest_threshold:

        text.set_color(
            "white"
        )

    else:

        text.set_color(
            "black"
        )


# =============================================================================
# Axis labels
# =============================================================================

ax.set_xlabel(
    "Model Reconstruction",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=16
)


ax.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Tick positions
# =============================================================================

ax.tick_params(
    axis="x",
    rotation=0,
    length=0
)


ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)


# =============================================================================
# Force x-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_xticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        X_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )

    label.set_horizontalalignment(
        "center"
    )

    label.set_multialignment(
        "center"
    )


# =============================================================================
# Force y-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        Y_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )


# =============================================================================
# Colorbar formatting
# =============================================================================

cbar = hm.collections[
    0
].colorbar


cbar.set_label(
    "% of Reconstruction Error",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


# =============================================================================
# Colorbar ticks
# =============================================================================

candidate_ticks = np.array(
    [
        vmin,
        5,
        10,
        15,
        20,
        30,
        40,
        50,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [
        f"{tick:.0f}%"
        for tick in nice_ticks
    ]
)


# =============================================================================
# Force all colorbar tick labels to same font and weight
# =============================================================================

for label in cbar.ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        CBAR_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cax.yaxis.set_label_position(
    "right"
)


cax.yaxis.set_ticks_position(
    "right"
)


# =============================================================================
# Strengthen colorbar outline
# =============================================================================

cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Save and display
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG to: "
    f"{png_path}"
)


print(
    f"Saved PDF to: "
    f"{pdf_path}"
)


plt.show()


plt.close(
    fig
)

<LOCAL_PATH>